# Scalabilité : combien d'utilisateurs faut-il ?

Troisième notebook de la série, après `train_cellid_llm.ipynb` (un entraînement) et
`train_cellid_week.ipynb` (sept jours). Celui-ci répond à **une seule question** :
*qu'est-ce que le modèle gagne quand on multiplie le nombre d'utilisateurs
d'entraînement ?* Il enchaîne un entraînement par **palier** (500 → 21 000
utilisateurs) et produit d'un coup tout le matériel de présentation : courbe de
scaling, intervalles de confiance appariés, coût GPU, et le détail de *qui* profite
de l'échelle.

## Le protocole, et pourquoi il est comme ça

| Choix | Raison |
|---|---|
| Les paliers sont **emboîtés** (`train.jsonl` est déjà mélangé : les 5 000 premiers utilisateurs contiennent les 2 000 premiers, et les 11 jours y sont uniformément représentés) | Un palier plus grand = strictement *plus* de données, jamais *d'autres* données. Sans ça, l'écart entre deux paliers mélange « taille » et « échantillon » — le défaut qui rend la comparaison entre jours du notebook semaine si délicate. |
| Le **jeu de test est le même pour tous les paliers** (`test_1000_users.jsonl`, jamais entraîné) | Tous les points de la courbe partagent la même baseline et les mêmes utilisateurs → les écarts se testent **par paires** (bootstrap apparié), ce qui est bien plus fin que des intervalles indépendants. |
| Le **vocabulaire et le tokenizer sont construits une fois**, sur l'union train ∪ test | Sinon un petit palier affronterait un softmax plus petit, et ses `top-k` seraient artificiellement flattés. Les `cell_id` qu'un petit palier ne voit jamais restent à leur initialisation : c'est précisément ce que la ligne « cellules rares » mesure. |
| La **validation porte sur les mêmes utilisateurs à tous les paliers** (préfixe fixe du fichier) | L'early stopping doit avoir le même bruit partout. Valider sur « tout le train » ferait porter la décision d'arrêt sur 500 utilisateurs à un palier et 21 000 à l'autre : deux règles d'arrêt différentes déguisées en une seule. |
| **Deux protocoles** : `iso_steps` (même nombre de pas d'optimisation partout) et `iso_epochs` (recette identique aux runs précédents, budget proportionnel aux données) | `iso_steps` isole l'apport du *nombre d'utilisateurs distincts* à compute constant — c'est la vraie courbe de scaling. `iso_epochs` répond à l'autre question, « et si on laisse converger ? », mais son coût croît linéairement avec le palier. |

## Ce qu'il produit

Un JSON par (protocole, palier) écrit au fil de l'eau — donc **reprenable** après une
déconnexion Colab — puis, sans GPU : le tableau de scaling, la loi d'échelle ajustée,
les intervalles de confiance appariés entre paliers consécutifs, le détail par groupe
de comportement, la courbe de démarrage à froid (accuracy selon la longueur du
contexte), les cellules rares, le coût GPU, et six graphiques.


In [ ]:
# ============================== CONFIGURATION ==============================
from pathlib import Path
import sys

# Ajout des dossiers "python" et répertoires courants au PYTHONPATH
for _p in ("python", ".", "/content", "/content/python"):
    _path = str(Path(_p).resolve())
    if Path(_p).exists() and _path not in sys.path:
        sys.path.insert(0, _path)

# --- DÉBUT DU BLOC GÉNÉRÉ (python/sync_notebook_fallbacks.py) ---
# ⚠️  NE PAS ÉDITER À LA MAIN : bloc régénéré par `make sync-notebook`.
# Chaque module de python/ est embarqué ici en clair pour que le notebook soit
# AUTONOME (Colab neuf, machine vierge : aucun fichier annexe à uploader). Le
# notebook n'écrit un module sur le disque que si son import échoue -- une
# installation normale du dépôt continue donc d'utiliser python/<module>.py.
# `tests/test_notebook_fallbacks.py` échoue si une copie ci-dessous diverge de
# son fichier source, pour qu'il n'existe qu'une seule source de vérité.
_EMBEDDED_MODULES = {
    'cellid_encoding': r'''"""Time-aware event encoding shared by the data-prep scripts and the training
notebook.

An "event" is one (cell_id, hour) pair from a user's daily trajectory. Hour is
the hour-of-day (0-23) derived from the raw timestamp (seconds since local
midnight, per CLAUDE.md) attached to that cell_id -- the timestamp follows the
cell_id it belongs to.
"""


def event_hour_from_seconds(ts_seconds):
    """Hour of day (0-23) from a timestamp in seconds since local midnight."""
    return (int(ts_seconds) // 3600) % 24


def hour_token(hour):
    """Dedicated tokenizer token for an hour-of-day, e.g. hour_token(6) == '<H06>'."""
    return f"<H{hour:02d}>"


HOUR_VOCAB = [hour_token(h) for h in range(24)]


def in_transition_window(hour, transition_windows):
    """True if `hour` falls in any (start_hour, end_hour, weight) window.
    Bounds are half-open [start_hour, end_hour) -- end excluded."""
    return any(start <= hour < end for start, end, _weight in transition_windows)


def event_weight(hour, transition_windows, base_weight=1.0):
    """Loss weight for an event at `hour`: the configured window's weight if
    `hour` falls inside one of `transition_windows`, else `base_weight`.
    `hour=None` (hour-less legacy data) always returns `base_weight`."""
    if hour is None:
        return base_weight
    for start, end, weight in transition_windows:
        if start <= hour < end:
            return weight
    return base_weight


def split_index(n_events, context_fraction):
    """Cutoff index for a context/target split over `n_events` events: at
    least 1, at most n_events - 1."""
    return max(1, min(n_events - 1, round(n_events * context_fraction)))


def make_chunks(n_events, chunk_len, chunk_stride):
    """Sliding-window (start, end, n_context_events) triples over event
    indices [0, n_events). `n_context_events` is how many of the window's
    leading events are overlap from the previous window (already trained on,
    excluded from the loss the second time). Windows are at most `chunk_len`
    events, advancing by `chunk_stride` each time."""
    if n_events <= chunk_len:
        return [(0, n_events, 0)]
    out, start = [], 0
    while start < n_events:
        end = min(start + chunk_len, n_events)
        ctx = 0 if start == 0 else chunk_len - chunk_stride
        if end - start > ctx:
            out.append((start, end, ctx))
        if start + chunk_len >= n_events:
            break
        start += chunk_stride
    return out


def event_token_positions(n_events, has_hours):
    """Index (relative to the first token *after* the prompt prefix) of each
    event's cell_id token in the flat sequence built by `encode_events`. When
    `has_hours` is True, each event is 2 tokens (hour then cell_id), so
    cell_id tokens sit at 1, 3, 5, ...; when False, each event is 1 token
    (cell_id only), so they sit at 0, 1, 2, ..."""
    step = 2 if has_hours else 1
    offset = 1 if has_hours else 0
    return [offset + i * step for i in range(n_events)]


def encode_events(cells, hours, cell_to_id, hour_to_id, prefix_ids,
                  transition_windows, n_masked_cells=0, base_weight=1.0):
    """Build (input_ids, labels, weights) for one sequence of events.

    cells: list[str] cell_id per event.
    hours: list[int] | None. If None, cell tokens only (legacy/no hour info) --
        no interleaved hour tokens, every unmasked position gets `base_weight`.
    cell_to_id / hour_to_id: dict[str, int] tokenizer vocab lookups.
    prefix_ids: list[int] tokens prepended before any event (e.g. a fixed
        prompt) -- always masked (label -100, weight base_weight).
    n_masked_cells: the first N events' cell labels are masked (-100) --
        used for chunk overlap, so a repeated context window isn't trained on
        twice.
    Returns three lists of equal length (one entry per token): input_ids,
    labels, weights.
    """
    ids = list(prefix_ids)
    labels = [-100] * len(prefix_ids)
    weights = [base_weight] * len(prefix_ids)
    for i, cell in enumerate(cells):
        hour = hours[i] if hours is not None else None
        if hour is not None:
            ids.append(hour_to_id[hour_token(hour)])
            labels.append(-100)
            weights.append(base_weight)
        ids.append(cell_to_id[cell])
        if i < n_masked_cells:
            labels.append(-100)
            weights.append(base_weight)
        else:
            labels.append(cell_to_id[cell])
            weights.append(event_weight(hour, transition_windows, base_weight))
    return ids, labels, weights
''',
    'user_groups': r'''"""Behavioural user groups for cross-training.

Rationale
---------
`cell_id` sequences look very different from one user to the next, but the
differences are not arbitrary: they are largely captured by *how often the user
changes physical site, as a function of the hour of day*. Two users who both
"stay put in the morning and move a lot late in the afternoon" are far more
informative about each other than two users picked at random -- which is
exactly what cross-training needs.

A group is therefore defined by a **site-change-rate profile over hour bands**:

    profile[b] = P(site root changes | the event falls in band b)

Measuring *site root* changes rather than *cell_id* changes matters. Per
CLAUDE.md a `cell_id` is `<tech letter><site root><zone digits>`, so the same
physical antenna appears under several `cell_id` values. On this dataset 12.4%
of all consecutive-event transitions are "same site root, different
radio/sector" -- i.e. the phone re-attached to another cell of the *same*
antenna without the user going anywhere. Counting those as movement would blur
the very distinction the groups are meant to capture, so `site_root()` strips
the technology letter and the zone digits before comparing.

Sparse bands are handled by shrinking each user's per-band rate toward the
global (train-set) rate for that band, with `shrinkage` pseudo-transitions of
prior weight -- a user with 3 night events does not get a wildly confident
night rate.

What the analysis found on 400_users_train.jsonl (4 bands, k=4)
---------------------------------------------------------------
Groups are numbered by increasing overall mobility, so the ordering is stable
and meaningful (G0 = most sedentary, G3 = most mobile in the morning):

    G0 "sédentaire"        28% of users  site-change 28/32/34/38% (sleep/morn/day/eve)
                           persistence 56% -- "repeat the last cell_id" is already strong
    G1 "régulier"          35%           31/47/46/45% -- flat, moderate mobility
                           persistence 42%
    G2 "actif après-midi"  14%           30/47/67/60% -- calm morning, very mobile
                           persistence 20%    from 16h to 20h (peak change rate 84%)
    G3 "actif le matin"    23%           35/67/65/43% -- very mobile 07h-13h, then
                           persistence 18%    settles down in the evening

G2 and G3 are near mirror images in time, and that shape is not an artefact of
sequence length: the (morning - evening) rate asymmetry is +0.244 for G3 vs
-0.135 for G2 while correlating only -0.28 with log(sequence length).

Why this changes the loss weighting
-----------------------------------
The single global transition window `(4, 6)` turns out to be *easier* than
average for all four groups (persistence 49-64% inside it vs 20-57% overall) --
so upweighting it was pushing gradient at positions the baseline already gets
right. And `(18, 20)` is only genuinely hard for G2; for G3 it is where the
user has just settled down for the evening (36% persistence vs 20% overall,
i.e. easier). `derive_group_windows()` replaces that one global guess with
per-group windows read off the data: the contiguous hours where a group's
`cell_id` persistence falls furthest below its own average, which is precisely
where the model has to reason instead of repeating.

This is the "keyed by group rather than a single global default" iteration that
CLAUDE.md anticipated.
"""
import re
from collections import Counter

import numpy as np

# ---------------------------------------------------------------- cell parsing
# <tech letter><site root><zone digits>; zone is 3 digits for 3G (<1|2><01-09>)
# and a single digit for 2G. See CLAUDE.md.
CELL_ID_RE = re.compile(r"^([BDUV])([A-Z]+?)((?:[12]\d{2})|\d)$")


def parse_cell_id(cell_id):
    """(tech_letter, site_root, zone_digits) or None if `cell_id` doesn't match
    the documented `<tech><root><zone>` shape."""
    m = CELL_ID_RE.match(cell_id)
    return m.groups() if m else None


def site_root(cell_id):
    """The physical-site part of a `cell_id` -- what actually has to change for
    the user to have *moved*. Falls back to the raw string if unparsable, so an
    unexpected id degrades to "its own site" instead of raising."""
    parsed = parse_cell_id(cell_id)
    return parsed[1] if parsed else cell_id


# ------------------------------------------------------------------ hour bands
# (name, start_hour, end_hour) with half-open bounds [start, end). A band whose
# start > end wraps midnight ("sleep" = 22h..07h). Four wide bands beat 5/6/8
# narrower ones on this dataset: every band keeps enough events to be estimated
# reliably, which raised the variance of per-user predictability explained by
# the group label from 0.65 (6 bands) to 0.69, and prefix-detectability from
# 85% to 86%.
DEFAULT_BANDS = [("sleep", 22, 7), ("morning", 7, 12), ("day", 12, 18), ("evening", 18, 22)]

DEFAULT_N_GROUPS = 4
DEFAULT_SHRINKAGE = 8.0


def band_of_hour(hour, bands=DEFAULT_BANDS):
    """Index of the band containing `hour`, honouring midnight-wrapping bands."""
    for i, (_name, start, end) in enumerate(bands):
        if start <= end:
            if start <= hour < end:
                return i
        elif hour >= start or hour < end:      # wraps midnight
            return i
    return 0


def band_transition_counts(cells, hours, bands=DEFAULT_BANDS):
    """(moves, transitions) per band for one user, counted on site-root changes.

    A "transition" is a consecutive pair of events; it is attributed to the band
    of the *later* event (the one being predicted). Returns two float arrays of
    length len(bands)."""
    n_bands = len(bands)
    moves = np.zeros(n_bands)
    totals = np.zeros(n_bands)
    if hours is None:
        return moves, totals
    roots = [site_root(c) for c in cells]
    for i in range(1, len(roots)):
        b = band_of_hour(hours[i], bands)
        totals[b] += 1
        moves[b] += roots[i] != roots[i - 1]
    return moves, totals


def fit_band_prior(users, bands=DEFAULT_BANDS):
    """Global per-band site-change rate over `users` -- the shrinkage target.
    Fit on the TRAIN split only, so test users never influence the geometry."""
    moves = np.zeros(len(bands))
    totals = np.zeros(len(bands))
    for _uid, cells, hours in users:
        m, t = band_transition_counts(cells, hours, bands)
        moves += m
        totals += t
    return moves / np.maximum(totals, 1.0)


def band_profile(cells, hours, prior, bands=DEFAULT_BANDS, shrinkage=DEFAULT_SHRINKAGE):
    """One user's per-band site-change rate, shrunk toward `prior`.

    `shrinkage` acts as that many pseudo-transitions already observed at the
    prior rate, so a band with few real transitions stays near the population
    rate instead of jumping to 0% or 100%.

    A band with no observed transitions *and* `shrinkage == 0` would otherwise
    be 0/0; it falls back to the prior rate for that band. Without this the
    profile would carry NaNs, every centroid distance would be NaN, and
    `GroupModel.assign` would silently put every user in group 0."""
    moves, totals = band_transition_counts(cells, hours, bands)
    numer = moves + shrinkage * prior
    denom = totals + shrinkage
    empty = denom == 0
    if empty.any():
        numer = np.where(empty, prior, numer)
        denom = np.where(empty, 1.0, denom)
    return numer / denom


# ---------------------------------------------------------------- group tokens
def group_token(group):
    """Dedicated tokenizer token for a behavioural group, e.g. '<G0>'."""
    return f"<G{group}>"


def group_vocab(n_groups=DEFAULT_N_GROUPS):
    return [group_token(g) for g in range(n_groups)]


# ------------------------------------------------------------------ the model
class GroupModel:
    """K-means over shrunk per-band site-change profiles, fitted on train users.

    Centroids are re-ordered by increasing mean site-change rate, so group 0 is
    always the most sedentary and group n-1 the most mobile whatever k-means'
    internal labelling was. Assignment is nearest centroid in Euclidean
    distance on the raw (un-standardised) profile: the bands are already
    commensurable rates in [0, 1], and standardising them made things worse --
    it inflates the sparse night band into the dominant axis (R^2 0.69 -> 0.43
    on this dataset)."""

    def __init__(self, centers, prior, bands, shrinkage, names=None):
        self.centers = np.asarray(centers, dtype=float)
        self.prior = np.asarray(prior, dtype=float)
        self.bands = list(bands)
        self.shrinkage = float(shrinkage)
        self.names = list(names) if names else [f"G{g}" for g in range(len(self.centers))]

    @property
    def n_groups(self):
        return len(self.centers)

    def profile(self, cells, hours):
        return band_profile(cells, hours, self.prior, self.bands, self.shrinkage)

    def assign(self, cells, hours):
        """Group of one user, from whatever slice of their day is passed in.

        Passing a *prefix* is what makes group detection legitimate at test
        time: the label comes only from events the model has already been
        given, never from the target it is being asked to predict."""
        if hours is None:
            return 0
        d = ((self.centers - self.profile(cells, hours)) ** 2).sum(axis=1)
        return int(np.argmin(d))

    def assign_all(self, users):
        return [self.assign(cells, hours) for _uid, cells, hours in users]


def fit_groups(users, n_groups=DEFAULT_N_GROUPS, bands=DEFAULT_BANDS,
               shrinkage=DEFAULT_SHRINKAGE, seed=42):
    """Fit a GroupModel on `users` (train split only). Requires scikit-learn."""
    from sklearn.cluster import KMeans

    prior = fit_band_prior(users, bands)
    X = np.array([band_profile(cells, hours, prior, bands, shrinkage)
                  for _uid, cells, hours in users])
    km = KMeans(n_clusters=n_groups, n_init=50, random_state=seed).fit(X)
    order = np.argsort(km.cluster_centers_.mean(axis=1))   # sedentary -> mobile
    return GroupModel(km.cluster_centers_[order], prior, bands, shrinkage)


# -------------------------------------------------- per-group loss weighting
def hour_persistence(users, labels, group, n_hours=24):
    """(same_cell_id, transitions) per hour for one group -- how often "repeat
    the previous cell_id" is correct at each hour. Uses raw `cell_id` equality,
    not site root: that is exactly the quantity the model's top-1 competes
    against."""
    same = np.zeros(n_hours)
    totals = np.zeros(n_hours)
    for (_uid, cells, hours), lab in zip(users, labels):
        if hours is None or lab != group:
            continue
        for i in range(1, len(cells)):
            h = hours[i] % n_hours
            totals[h] += 1
            same[h] += cells[i] == cells[i - 1]
    return same, totals


def derive_group_windows(users, labels, n_groups, min_support=25, min_window_events=60,
                        min_deficit=0.05, weight_gain=2.0, max_weight=4.0):
    """Per-group transition windows read off the training data.

    For each group, find the maximal runs of consecutive hours where that
    group's `cell_id` persistence sits at least `min_deficit` *below* the
    group's own average persistence -- the hours where the "repeat the last
    cell_id" baseline breaks down and the model actually has to reason.

    Hours with fewer than `min_support` observed transitions are ignored (too
    noisy to trust), and a candidate run needs `min_window_events` transitions
    in total to be kept, so a weight is never derived from a handful of events.

    The weight scales with how much harder the window is than the group's
    average::

        ratio  = group_mean_persistence / window_persistence
        weight = clip(1 + weight_gain * (ratio - 1), 1.0, max_weight)

    Returns {group: [(start_hour, end_hour, weight), ...]}. A group whose
    persistence is flat across the day legitimately gets an **empty** list: for
    G0 on this dataset "repeat the last cell_id" is uniformly strong (56%), so
    there is no hour worth upweighting and a flat weight of 1.0 is correct.
    """
    windows = {}
    for g in range(n_groups):
        same, totals = hour_persistence(users, labels, g)
        ok = totals >= min_support
        if not ok.any():
            windows[g] = []
            continue
        base = same[ok].sum() / totals[ok].sum()
        persistence = np.where(ok, same / np.maximum(totals, 1.0), np.inf)
        found = []
        h = 0
        while h < len(totals):
            if base - persistence[h] > min_deficit:
                start = h
                while h < len(totals) and base - persistence[h] > min_deficit:
                    h += 1
                n_events = totals[start:h].sum()
                if n_events >= min_window_events:
                    acc = same[start:h].sum() / n_events
                    ratio = base / max(acc, 1e-6)
                    w = float(np.clip(1.0 + weight_gain * (ratio - 1.0), 1.0, max_weight))
                    found.append((int(start), int(h), round(w, 2)))
            else:
                h += 1
        windows[g] = found
    return windows


def windows_for(group, group_windows, fallback=()):  # noqa: D401
    """Windows of `group`, or `fallback` when the group has none configured."""
    return group_windows.get(group, list(fallback))


# --------------------------------------------------------------- description
def describe_groups(users, labels, model, n_groups):
    """Per-group summary rows for printing: size, band profile, persistence."""
    rows = []
    counts = Counter(labels)
    for g in range(n_groups):
        idx = [i for i, lab in enumerate(labels) if lab == g]
        if not idx:
            rows.append({"group": g, "n": 0})
            continue
        profiles = np.array([model.profile(users[i][1], users[i][2]) for i in idx])
        pers, lens = [], []
        for i in idx:
            cells = users[i][1]
            lens.append(len(cells))
            if len(cells) > 1:
                pers.append(np.mean([cells[j] == cells[j - 1] for j in range(1, len(cells))]))
        rows.append({
            "group": g,
            "n": counts[g],
            "share": counts[g] / len(labels) if labels else 0.0,
            "profile": profiles.mean(axis=0),
            "persistence": float(np.mean(pers)) if pers else float("nan"),
            "mean_len": float(np.mean(lens)),
        })
    return rows
''',
    'generate_sample_users': r'''#!/usr/bin/env python3
"""Génère des fichiers <N>_users_{train,test}.jsonl synthétiques, au même format
que le jeu réel produit par `make data-train`.

Sert de **repli autonome** : le notebook peut tourner de bout en bout sans aucun
fichier de données externe. Les séquences portent donc, comme les données réelles :

  * des `cell_id` au format documenté dans CLAUDE.md -- `<lettre techno><site
    root><chiffres de zone>` -- de sorte que plusieurs `cell_id` partagent le même
    site physique (c'est ce que `user_groups.site_root()` compare) ;
  * une **heure** par événement (champ `hours`), sans laquelle le cross-training
    par groupe de comportement se désactive faute de signal horaire ;
  * quatre **archétypes de mobilité** calqués sur les groupes observés dans les
    vraies données, pour que le regroupement ait quelque chose à retrouver :
    sédentaire / régulier / actif l'après-midi / actif le matin.

Usage :
    python generate_sample_users.py --n-users 500 [--seed 42] [--out-dir DIR]
    python generate_sample_users.py --n-train 400 --n-test 100 --out-dir data/dataset_for_training
"""
import argparse
import json
import random
from pathlib import Path

SYSTEM_PROMPT = ("You are an AI that generates a user's cell-visit trajectory. "
                 "Output format: User <USER_ID> | <N_RECORDS> events | "
                 "<CELL_ID> <CELL_ID> ...")
USER_PROMPT = "Here is the sequence for this user."

# Archétypes : (nom, probabilité de changer de site par tranche, événements/heure).
# Les tranches sont (nuit 22-07, matin 07-12, jour 12-18, soir 18-22), dans l'ordre
# de CONFIG["group_bands"]. Les valeurs reprennent les profils mesurés sur les 400
# vrais utilisateurs (cf. python/user_groups.py) -- y compris le fait que les
# sédentaires émettent nettement plus d'événements par heure.
ARCHETYPES = [
    ("sedentaire",     (0.28, 0.32, 0.34, 0.38), 10.0, 0.30),
    ("regulier",       (0.31, 0.47, 0.46, 0.45),  9.0, 0.22),
    ("actif_apresmidi",(0.30, 0.47, 0.67, 0.60),  4.6, 0.15),
    ("actif_matin",    (0.35, 0.67, 0.65, 0.43),  4.8, 0.15),
]
BANDS = ((22, 7), (7, 12), (12, 18), (18, 22))


def band_of_hour(hour):
    for i, (start, end) in enumerate(BANDS):
        if start <= end:
            if start <= hour < end:
                return i
        elif hour >= start or hour < end:
            return i
    return 0


def load_real_vocab():
    """Vocabulaire repris des fichiers réels s'ils sont là, sinon []."""
    vocab = set()
    for path in ("100_users_train.jsonl", "100_users_test.jsonl"):
        try:
            with open(path, encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    if "cells" in row:
                        vocab.update(row["cells"])
                    else:
                        content = row["conversations"][-1]["content"]
                        vocab.update(content.split("|", 2)[2].split())
        except (FileNotFoundError, KeyError, ValueError):
            pass
    return sorted(vocab)


def synthetic_vocab(n_sites=110):
    """Vocabulaire au format CLAUDE.md : pour chaque site, des cellules 2G
    (`B<root><1-4>`) et 3G (`U<root><1|2><01-04>`). Plusieurs `cell_id` par site
    physique, exactement comme dans les vraies données."""
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sites = {}
    for i in range(n_sites):
        root = "KV" + letters[i // 26 % 26] + letters[i % 26] + letters[(i * 7) % 26]
        cells = [f"B{root}{s}" for s in range(1, 5)]
        cells += [f"U{root}{z}{s:02d}" for z in (1, 2) for s in range(1, 5)]
        sites[root] = cells
    return sites


def sites_from_vocab(vocab):
    """Regroupe un vocabulaire plat par site physique (préfixe sans techno/zone)."""
    import re
    pat = re.compile(r"^([BDUV])([A-Z]+?)((?:[12]\d{2})|\d)$")
    sites = {}
    for cell in vocab:
        m = pat.match(cell)
        root = m.group(2) if m else cell
        sites.setdefault(root, []).append(cell)
    return sites


def make_sequence(rng, sites, archetype):
    """Une journée : répertoire de sites restreint, mobilité dépendant de l'heure.

    Un « déplacement » change de site ; sinon on reste sur place, en réémettant
    parfois une AUTRE cellule du même site (changement techno/secteur) -- c'est
    ce bruit qui rend `site_root()` indispensable côté analyse."""
    _name, move_probs, per_hour, same_site_switch = archetype
    roots = rng.sample(sorted(sites), k=rng.randint(12, 24))
    home = roots[0]
    start = rng.choices([0, 1, 6, 7, 8, 9, 10], weights=[15, 10, 12, 18, 16, 15, 14])[0]
    end = rng.choices([15, 16, 17, 18, 19, 22, 23], weights=[12, 14, 16, 12, 12, 16, 18])[0]
    if end <= start:
        end = min(23, start + 6)

    cells, hours = [], []
    current = home
    current_cell = rng.choice(sites[home])
    for hour in range(start, end + 1):
        n_events = max(1, int(rng.gauss(per_hour, per_hour * 0.35)))
        p_move = move_probs[band_of_hour(hour)]
        for _ in range(n_events):
            if rng.random() < p_move:
                # déplacement : nouveau site, donc nouvelle cellule
                current = rng.choice([r for r in roots if r != current])
                current_cell = rng.choice(sites[current])
            elif rng.random() < same_site_switch:
                # sur place, mais le téléphone se raccroche à une AUTRE cellule du
                # même site (changement techno/secteur) : le cell_id change sans
                # déplacement -- c'est ce bruit qui rend site_root() indispensable
                others = [c for c in sites[current] if c != current_cell]
                if others:
                    current_cell = rng.choice(others)
            # sinon : on ne bouge pas et on réémet EXACTEMENT la même cellule,
            # ce qui fait de « recopier le cell_id précédent » une baseline forte
            cells.append(current_cell)
            hours.append(hour)
    # borne de longueur, comme dans les vraies données (19 à 200 événements)
    if len(cells) > 200:
        cells, hours = cells[:200], hours[:200]
    while len(cells) < 19:
        cells.append(cells[-1] if cells else rng.choice(sites[home]))
        hours.append(hours[-1] if hours else start)
    return cells, hours


def make_line(user_id, cells, hours, with_hours=True):
    """Même schéma que python/format_for_train.py : champs structurés
    `cells`/`hours` (ce que le notebook lit) + le texte `conversations`."""
    row = {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant",
             "content": f"User {user_id} | {len(cells)} events | {' '.join(cells)}"},
        ],
        "user_id": str(user_id),
        "cells": cells,
    }
    if with_hours:
        row["hours"] = hours
    return json.dumps(row, ensure_ascii=False)


def generate(n_train, n_test, seed, out_dir, with_hours=True, stem=None):
    rng = random.Random(seed)
    vocab = load_real_vocab()
    if vocab:
        sites = sites_from_vocab(vocab)
    else:
        sites = synthetic_vocab()
        print("⚠️  Fichiers 100_users introuvables : vocabulaire synthétique utilisé "
              f"({len(sites)} sites au format CLAUDE.md).")

    n_total = n_train + n_test
    user_ids = rng.sample(range(19_000_000, 21_000_000), n_total)
    lines = []
    for i, uid in enumerate(user_ids):
        arch = ARCHETYPES[i % len(ARCHETYPES)]     # les 4 archétypes équirépartis
        cells, hours = make_sequence(rng, sites, arch)
        lines.append(make_line(uid, cells, hours, with_hours))
    rng.shuffle(lines)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = stem or f"{n_train}_users"
    written = []
    for name, chunk in ((f"{stem}_train.jsonl", lines[:n_train]),
                        (f"{stem}_test.jsonl", lines[n_train:])):
        dest = out_dir / name
        dest.write_text("\n".join(chunk) + "\n", encoding="utf-8")
        print(f"{dest} : {len(chunk)} utilisateurs")
        written.append(dest)
    return written


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--n-users", type=int, default=None,
                        help="nombre total d'utilisateurs (split 80/20 train/test)")
    parser.add_argument("--n-train", type=int, default=None, help="nombre exact de lignes train")
    parser.add_argument("--n-test", type=int, default=None, help="nombre exact de lignes test")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out-dir", type=Path, default=Path("."),
                        help="dossier de sortie (défaut : dossier courant)")
    parser.add_argument("--stem", default=None,
                        help="préfixe des fichiers (défaut : <n_train>_users)")
    parser.add_argument("--no-hours", action="store_true",
                        help="ne pas émettre le champ hours (désactive le mode heure du notebook)")
    args = parser.parse_args()

    if args.n_train is None or args.n_test is None:
        total = args.n_users if args.n_users is not None else 500
        n_train = int(total * 0.8)
        n_test = total - n_train
    else:
        n_train, n_test = args.n_train, args.n_test

    generate(n_train, n_test, args.seed, args.out_dir,
             with_hours=not args.no_hours, stem=args.stem)


if __name__ == "__main__":
    main()
''',
}
# --- FIN DU BLOC GÉNÉRÉ ---
# Écriture des modules embarqués UNIQUEMENT s'ils ne sont pas importables : sur un
# dépôt normalement installé, python/ est dans sys.path et rien n'est écrit.
_written = []
for _name, _code in _EMBEDDED_MODULES.items():
    try:
        __import__(_name)
    except ImportError:
        Path(f"{_name}.py").write_text(_code, encoding="utf-8")
        _written.append(_name)
if _written:
    import importlib
    importlib.invalidate_caches()
    print("Modules non trouvés — générés automatiquement dans le répertoire courant ✅ : "
          + ", ".join(f"{m}.py" for m in _written))

# ============ HYPERPARAMÈTRES — IDENTIQUES POUR TOUS LES PALIERS ============
# C'est la condition de l'expérience : si un réglage change entre deux paliers,
# l'écart mesuré n'est plus l'effet de la taille du jeu de données.
CONFIG = {
    # Modèle de base — décommentez celui voulu :
    # "model_name": "mistralai/Mistral-7B-v0.1",    # 7B : GPU obligatoire, 4-bit (QLoRA), ~8-10x le temps
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",     # 0.5B : le bon choix pour une échelle de 7 paliers

    "output_dir": "outputs_final",
    "seed": 42,

    # Token Hugging Face (optionnel : Qwen2.5 est public).
    # ⚠️ Si vous partagez ce notebook, videz cette valeur et utilisez les secrets Colab.
    "hf_token": "",

    # --- LoRA / optimisation (identiques aux notebooks précédents) ---
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "learning_rate": 1.5e-4,
    "weight_decay": 0.01,
    "warmup_steps": 20,
    "max_seq_len": 512,          # longueur max à l'évaluation (séquence complète)
    "chunk_len": 128,            # fenêtres d'entraînement, en nombre d'événements
    "chunk_stride": 64,
    # Fraction de la séquence donnée en contexte : le modèle n'est entraîné QUE sur
    # ce préfixe et doit prédire la suite, jamais vue. Appliqué au train ET au test.
    "context_fraction": 0.8,
    "prefix_text": "Events:",

    # --- Arrêt (protocole iso_epochs, cf. SCALE_CONFIG pour iso_steps) ---
    "early_stop_patience": 4,    # epochs sans progrès
    "max_epochs": 20,            # AUSSI l'horizon du scheduler cosine : ne pas baisser
                                 # pour « aller plus vite », cela change le régime d'apprentissage

    # Sur-échantillonnage des fenêtres touchant un créneau de transition
    "transition_chunk_oversample": 3.0,
    # Créneau GLOBAL de repli (utilisé seulement si groupes désactivés ou pas d'heures)
    "transition_windows": [(4, 6, 2.0), (18, 20, 4.0)],

    # ============ CROSS-TRAINING PAR GROUPE DE COMPORTEMENT ============
    # Les centroïdes sont réajustés SUR LE TRAIN DE CHAQUE PALIER : un groupe doit
    # rester dérivable des seules données dont on dispose à ce palier-là. La
    # stabilité des groupes quand N grandit est elle-même un résultat de scalabilité
    # (colonne "détect." du tableau de synthèse).
    "use_groups": True,
    "n_groups": 4,
    "group_bands": [("sleep", 22, 7), ("morning", 7, 12), ("day", 12, 18), ("evening", 18, 22)],
    "group_shrinkage": 8.0,
    "group_windows": None,              # None => dérivés du train du palier
    "group_window_min_deficit": 0.05,
    "group_window_weight_gain": 2.0,
    "group_window_max_weight": 4.0,
    "group_detect_fraction": 0.5,

    # --- Guards (protection machine) ---
    "min_free_ram_gb": 2.0,
    "max_ram_used_fraction": 0.90,
    "min_free_disk_gb": 5.0,
    "ram_check_every_steps": 10,
    "save_total_limit": 1,
}


# ============ PILOTAGE DE L'ÉTUDE DE SCALABILITÉ ============
SCALE_CONFIG = {
    # --- Le plan d'expérience -------------------------------------------------
    # Chaque entrée = un protocole et les paliers auxquels on l'applique. Les jobs
    # sont exécutés du plus petit au plus grand : si la session tombe, le bas de la
    # courbe est déjà acquis. Un palier plus grand que le fichier est ignoré avec
    # un avertissement (le dernier palier vaut alors « tout le fichier »).
    "runs": [
        {"protocol": "iso_steps",  "tiers": [5000, 10000, 15000, 21000]},
    ],

    # --- Protocole iso_steps : même compute pour tous les paliers -------------
    # Tous les paliers reçoivent EXACTEMENT le même budget de pas d'optimisation,
    # le même warmup et le même horizon cosine. La seule chose qui change est le
    # nombre d'utilisateurs distincts dans lesquels les batches sont tirés : c'est
    # cela, et rien d'autre, que la courbe mesure.
    # 1000 pas ~ le pic du run de référence à 500 utilisateurs (epoch 9 x 67 pas).
    "max_steps": 1000,
    "eval_every_steps": 150,     # validation tous les N pas (une "epoch" ne veut plus
                                 # rien dire : elle vaut 67 pas à 500 users, 2815 à 21 000)
    "early_stop_evals": 4,       # arrêt après N validations sans progrès (= 600 pas)

    # --- Validation (IDENTIQUE à tous les paliers) ----------------------------
    # Sous-échantillon FIXE pris en tête du fichier train : comme les paliers sont
    # emboîtés, ces utilisateurs appartiennent au train de TOUS les paliers. Doit
    # rester <= au plus petit palier (vérifié au préflight).
    "val_subsample": 250,

    # --- Fichiers -------------------------------------------------------------
    "data_dirs": [
        "/content",
        "/content/data",
        "/content/drive/MyDrive/cellid_scaling",
        "data/data_nino_15days/merged",
        "data/dataset_for_training/merged",
        "data",
        ".",
    ],
    "train_file": "train.jsonl",
    "test_file": "test_1000_users.jsonl",

    # --- Résultats ------------------------------------------------------------
    "use_drive": True,
    "drive_results_dir": "/content/drive/MyDrive/cellid_scaling/results",
    "local_results_dir": "outputs_final/results",
    "force_rerun": [],           # ex. ["iso_steps_05000"] pour refaire un job terminé

    # --- Ce qu'on mesure ------------------------------------------------------
    # Fractions de contexte évaluées sur le test : la courbe « démarrage à froid ».
    # C'est là que l'échelle devrait payer le plus — un modèle nourri de 21 000
    # utilisateurs doit deviner mieux QUAND L'HISTORIQUE DU CLIENT EST COURT.
    # Réduit à [0.3] : deux points (30% et la référence 80%) suffisent à porter
    # le message « l'échelle paie quand l'historique est court ». Chaque fraction
    # en plus est une passe d'évaluation complète sur les 1000 utilisateurs de test.
    "test_context_fractions": [0.3],
    # Seuil « cellule rare » : un cell_id vu moins de N fois dans le train DU PALIER.
    "rare_cell_threshold": 50,
    # Diagnostics lourds (ablations groupe/heure, éval globale) : ~13 passes
    # d'évaluation supplémentaires. "extremes" = premier et dernier palier de chaque
    # protocole ; sinon une liste de paliers, ou [] pour aucun.
    "diagnostics_tiers": [],
    "eval_batch_size": 8,
    "train_batch_size": None,    # None => automatique selon le device
    "grad_accum": None,
    "save_adapters": False,      # ~550 Mo par palier sur un 7B : off par défaut

    # --- Budget ---------------------------------------------------------------
    # Calibration : mesure sec/pas et sec/évaluation sur le plus petit palier, puis
    # projette le planning complet AVANT de lancer quoi que ce soit.
    "calibrate": True,
    "calibration_steps": 12,
    # Avertissement si la projection dépasse ce budget (en minutes). None = aucun.
    "time_budget_min": 240,

    # Rognage POUR TESTER LE NOTEBOOK rapidement (ex. 60). None en usage normal.
    "limit_users": None,
}

JOBS = [(r["protocol"], t) for r in SCALE_CONFIG["runs"] for t in sorted(r["tiers"])]

def job_name(protocol, tier):
    """Identifiant stable d'un job — sert de nom de fichier de résultat."""
    return f"{protocol}_{tier:05d}"

print(f"Modèle  : {CONFIG['model_name']}")
print(f"Plan    : {len(JOBS)} entraînements")
for r in SCALE_CONFIG["runs"]:
    print(f"  {r['protocol']:<10} → paliers {', '.join(str(t) for t in sorted(r['tiers']))}")
print(f"Test    : {SCALE_CONFIG['test_file']} (le MÊME pour tous les paliers)")


In [ ]:
# ==================== DÉTECTION ENVIRONNEMENT (local / Colab / Drive) ====================
import os, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab détecté — installation des dépendances…")
    # torchao préinstallé sur Colab (0.10) est incompatible avec peft récent -> on le retire
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.45", "peft>=0.15", "datasets>=3.0",
                    "accelerate>=1.0", "psutil", "bitsandbytes"], check=True)

# --- Drive monté MAINTENANT, pas au moment d'écrire ---
# L'authentification Drive est interactive : la déclencher au bout de deux heures
# d'entraînement, c'est la rater et perdre la courbe.
RESULTS_DIR = Path(SCALE_CONFIG["local_results_dir"])
if IN_COLAB and SCALE_CONFIG["use_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = Path(SCALE_CONFIG["drive_results_dir"])
        print("Drive monté — les résultats survivront à une déconnexion.")
    except Exception as exc:
        print(f"⚠️  Drive non monté ({exc}) — repli sur {RESULTS_DIR} (perdu si la session tombe).")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Résultats : {RESULTS_DIR.resolve()}")

# --- Authentification Hugging Face (facultative, le modèle est public) ---
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or CONFIG["hf_token"]
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Authentifié sur Hugging Face Hub")

import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"GPU : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} Go)")
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float32   # fp32 : le plus stable sur Apple Silicon
else:
    DEVICE, DTYPE = "cpu", torch.float32

if DEVICE != "cuda":
    torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))

print(f"Environnement : {'Google Colab' if IN_COLAB else 'local'} | device={DEVICE} | dtype={DTYPE}")
if DEVICE != "cuda":
    print("⚠️  Une échelle de plusieurs paliers hors GPU n'est pas réaliste : lancez ce notebook")
    print("    sur Colab avec un GPU (Exécution > Modifier le type d'exécution).")


In [ ]:
# ==================== PRÉFLIGHT : RESSOURCES, FICHIERS, COHÉRENCE DU PLAN ====================
# Tout ce qui peut échouer doit échouer ICI. Un plan incohérent découvert au 6e
# palier coûte les cinq heures déjà calculées.
import shutil, psutil, json

def _search_dirs():
    seen, dirs = set(), []
    for d in SCALE_CONFIG["data_dirs"]:
        p = Path(d)
        if p.is_dir() and p.resolve() not in seen:
            seen.add(p.resolve())
            dirs.append(p)
    return dirs

def find_file(filename):
    """Le fichier `filename`, cherché à plat et sur un niveau de sous-dossier."""
    found = {}
    for d in _search_dirs():
        for path in list(d.glob(filename)) + list(d.glob(f"*/{filename}")):
            found[path.resolve()] = path
    paths = sorted(found.values(), key=lambda p: str(p))
    if not paths:
        raise FileNotFoundError(
            f"Fichier '{filename}' introuvable dans : "
            + ", ".join(str(d) for d in _search_dirs())
            + ". Ajoutez son dossier en tête de SCALE_CONFIG['data_dirs'].")
    if len(paths) > 1:
        raise ValueError(f"{len(paths)} fichiers '{filename}' : "
                         + ", ".join(str(p) for p in paths)
                         + ". Affinez SCALE_CONFIG['data_dirs'].")
    return paths[0]

def count_lines(path):
    with open(path, "rb") as f:
        return sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1 << 20), b""))

def preflight():
    vm = psutil.virtual_memory()
    free_ram_gb = vm.available / 1e9
    free_disk_gb = shutil.disk_usage(".").free / 1e9
    print(f"RAM libre : {free_ram_gb:.1f} Go ({vm.percent:.0f}% utilisée) | "
          f"Disque libre : {free_disk_gb:.1f} Go")
    if free_ram_gb < CONFIG["min_free_ram_gb"]:
        raise RuntimeError(f"RAM libre insuffisante (< {CONFIG['min_free_ram_gb']} Go).")
    if free_disk_gb < CONFIG["min_free_disk_gb"]:
        raise RuntimeError(f"Espace disque insuffisant (< {CONFIG['min_free_disk_gb']} Go).")

    if CONFIG.get("use_groups"):
        try:
            import user_groups            # noqa: F401
            import sklearn.cluster        # noqa: F401
        except ImportError as exc:
            raise ImportError(
                f"CONFIG['use_groups'] est True mais l'import a échoué ({exc}). "
                "Installez scikit-learn, ou mettez use_groups=False.") from exc

    files = {"train": find_file(SCALE_CONFIG["train_file"]),
             "test": find_file(SCALE_CONFIG["test_file"])}
    n_avail = count_lines(files["train"])
    for k, p in files.items():
        print(f"  {k:<6} {str(p):<64} ({p.stat().st_size / 1e6:.1f} Mo)")
    print(f"  → {n_avail} utilisateurs disponibles dans le fichier d'entraînement")

    # --- cohérence du plan ---------------------------------------------------
    tiers = sorted({t for _, t in JOBS})
    too_big = [t for t in tiers if t > n_avail]
    if too_big:
        print(f"  ⚠️  Paliers au-delà du fichier ({n_avail}) : {too_big} → ramenés à {n_avail}.")
    val_k = SCALE_CONFIG["val_subsample"]
    smallest = min(tiers)
    if val_k is not None and val_k > smallest:
        raise ValueError(
            f"val_subsample={val_k} dépasse le plus petit palier ({smallest}). Les "
            "utilisateurs de validation doivent appartenir au train de TOUS les paliers, "
            "sinon l'early stopping ne se compare plus d'un palier à l'autre. "
            f"Mettez val_subsample <= {smallest}, ou retirez le palier {smallest}.")

    print(f"\nPlan ({len(JOBS)} entraînements) :")
    for protocol, tier in JOBS:
        state = "déjà fait" if (RESULTS_DIR / f"{job_name(protocol, tier)}.json").exists() else "à faire"
        print(f"  {job_name(protocol, tier):<18} {min(tier, n_avail):>6} utilisateurs   [{state}]")
    print("\nPréflight OK ✅")
    return files, n_avail

FILES, N_AVAILABLE = preflight()


In [ ]:
# ==================== CHARGEMENT (UNE SEULE FOIS) ====================
# Le fichier d'entraînement complet est lu une fois et gardé en RAM : chaque palier
# n'est qu'une TRANCHE INITIALE de cette liste. C'est ce qui rend les paliers
# emboîtés par construction — le palier 5 000 contient exactement les 2 000 premiers
# utilisateurs du palier 2 000, plus 3 000 autres.
# Coût : ~40 Mo une fois les cell_id internés (369 chaînes distinctes partagées
# entre 1,5 million d'événements, au lieu d'autant d'objets).
from collections import Counter
from cellid_encoding import split_index, make_chunks

_INTERN = {}

def _intern(cell):
    got = _INTERN.get(cell)
    if got is None:
        got = _INTERN[cell] = sys.intern(cell)
    return got

def load_users(path, limit=None):
    """Parse le JSONL -> [(user_id, [cell_id, ...], [heure, ...] | None), ...].
    Si une ligne porte les champs structurés "cells"/"hours" (python/format_for_train.py),
    ils sont utilisés directement. Sinon on retombe sur le texte "assistant" —
    heures = None -> pas d'heure interleavée."""
    users = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(users) >= limit:
                break
            row = json.loads(line)
            if "cells" in row:
                uid = str(row.get("user_id", "?"))
                cells = [_intern(c) for c in row["cells"]]
                hours = row.get("hours")
            else:
                content = row["conversations"][-1]["content"]
                head, _, seq = content.split("|", 2)
                uid = head.split()[1]
                cells = [_intern(c) for c in seq.split()]
                hours = None
            users.append((uid, cells, hours))
    return users

def split_index_for(cells, context_fraction):
    return split_index(len(cells), context_fraction) if context_fraction is not None else 1

def baseline_persistence(users, context_fraction=None):
    """Prédire « même cell_id que le précédent » — la référence à battre. Sur le jeu
    de test FIXE, cette valeur est la même pour tous les paliers : c'est la ligne
    horizontale de tous les graphiques."""
    tot = ok = 0
    for _, s, _ in users:
        min_j = split_index_for(s, context_fraction)
        for i in range(1, len(s)):
            if i < min_j:
                continue
            tot += 1
            ok += s[i] == s[i - 1]
    return ok / tot if tot else float("nan")

def baseline_persistence_breakdown(users, context_fraction, transition_windows):
    """Comme baseline_persistence, mais séparément dans / hors créneaux de transition.
    `transition_windows` : liste globale (debut, fin, poids) OU callable ui -> liste."""
    win_of = transition_windows if callable(transition_windows) else (lambda _i: transition_windows)
    all_wins = sorted({(s, e) for i in range(len(users)) for s, e, _ in win_of(i)})
    tot = ok = tot_in = ok_in = tot_out = ok_out = 0
    win_hits = {w: 0 for w in all_wins}
    win_tots = {w: 0 for w in all_wins}
    for ui, (_, s, hours) in enumerate(users):
        wins = win_of(ui)
        min_j = split_index_for(s, context_fraction)
        for i in range(1, len(s)):
            if i < min_j:
                continue
            tot += 1
            hit = s[i] == s[i - 1]
            ok += hit
            if hours is not None:
                h = hours[i]
                in_any = False
                for (start, end, _) in wins:
                    if start <= h < end:
                        win_tots[(start, end)] += 1
                        win_hits[(start, end)] += hit
                        in_any = True
                        break
                if in_any:
                    tot_in += 1; ok_in += hit
                else:
                    tot_out += 1; ok_out += hit
    nan = float("nan")
    res = {"overall": ok / tot if tot else nan,
           "in_window": ok_in / tot_in if tot_in else nan,
           "out_window": ok_out / tot_out if tot_out else nan,
           "window_breakdown": {}}
    for w in all_wins:
        wtot = win_tots[w]
        res["window_breakdown"][w] = {"acc": win_hits[w] / wtot if wtot else nan, "n": wtot}
    return res

# --- lecture -----------------------------------------------------------------
LIMIT = SCALE_CONFIG["limit_users"]
ALL_TRAIN = load_users(FILES["train"], LIMIT)
TEST_USERS = load_users(FILES["test"], None if LIMIT is None else max(4, LIMIT // 4))
TIERS = sorted({min(t, len(ALL_TRAIN)) for _, t in JOBS})
if LIMIT is not None:
    print(f"⚠️  MODE TEST : fichier tronqué à {LIMIT} utilisateurs — résultats non représentatifs.\n")

# --- vocabulaire PARTAGÉ : l'union train ∪ test, indépendante du palier -------
# C'est ce qui rend les top-k comparables entre paliers : sinon un petit palier
# affronterait un softmax plus petit, donc une tâche plus facile, et la courbe de
# scaling mesurerait en partie la taille du softmax.
VOCAB = sorted({c for _, cells, _ in ALL_TRAIN + TEST_USERS for c in cells})
HAS_HOURS = any(hours is not None for _, _, hours in ALL_TRAIN + TEST_USERS)
TEST_VOCAB = {c for _, cells, _ in TEST_USERS for c in cells}

print(f"Train : {len(ALL_TRAIN)} utilisateurs | Test : {len(TEST_USERS)} (fixe, jamais entraînés)")
print(f"Vocabulaire PARTAGÉ (union train ∪ test) : {len(VOCAB)} cell_id")
print("Heures disponibles : "
      + ("oui" if HAS_HOURS else "NON — tokens heure, groupes et pondération désactivés"))
if not HAS_HOURS:
    print("   ⚠️  Vos JSONL ne portent pas de timestamps : le modèle entraîné sera la variante")
    print("       amputée (pas de <Hxx>, pas de <Gk>). Régénérez-les avec python/format_for_train.py.")

# --- ce que chaque palier apporte, AVANT toute seconde de GPU ----------------
# Ce tableau est déjà de la matière de présentation : il montre que la couverture
# du vocabulaire sature bien avant l'accuracy, donc que le gain d'échelle au-delà
# du point de saturation ne vient PAS de « découvrir de nouvelles cellules ».
RARE_T = SCALE_CONFIG["rare_cell_threshold"]
TIER_STATS = {}
print(f"\n{'palier':>7} {'événem.':>10} {'fenêtres':>9} {'vocab':>6} {'couv. test':>11} "
      f"{'cell. rares':>12} {'moy. len':>9}")
for n in TIERS:
    sub = ALL_TRAIN[:n]
    freq = Counter(c for _, cells, _ in sub for c in cells)
    lens = [len(cells) for _, cells, _ in sub]
    windows = sum(len(make_chunks(split_index(L, CONFIG["context_fraction"]),
                                  CONFIG["chunk_len"], CONFIG["chunk_stride"])) for L in lens)
    rare = {c for c in VOCAB if freq.get(c, 0) < RARE_T}
    TIER_STATS[n] = {"n_users": n, "n_events": sum(lens), "n_windows": windows,
                     "vocab": len(freq), "test_coverage": len(TEST_VOCAB & set(freq)) / len(TEST_VOCAB),
                     "n_rare": len(rare), "mean_len": sum(lens) / len(lens)}
    s = TIER_STATS[n]
    print(f"{n:>7} {s['n_events']:>10,} {windows:>9,} {s['vocab']:>6} "
          f"{s['test_coverage']:>10.1%} {s['n_rare']:>7}/{len(VOCAB):<4} {s['mean_len']:>9.0f}")

REF_FRAC = CONFIG["context_fraction"]
BASELINE_TEST = baseline_persistence(TEST_USERS, REF_FRAC)
N_TEST_PRED = sum(max(0, len(c) - split_index_for(c, REF_FRAC)) for _, c, _ in TEST_USERS)
print(f"\nBaseline persistance sur le test (contexte {REF_FRAC:.0%}) : {BASELINE_TEST:.1%} "
      f"sur n={N_TEST_PRED} prédictions.")
print("Le test étant IDENTIQUE à tous les paliers, cette valeur est la même partout :")
print("c'est la ligne horizontale de la courbe de scaling, et les écarts entre paliers")
print("se testent par bootstrap APPARIÉ sur les mêmes utilisateurs (cf. cellule SYNTHÈSE).")
print(f"« Cellule rare » = vue moins de {RARE_T} fois dans le train DU PALIER.")


In [ ]:
# ==================== TOKENIZER PARTAGÉ + FABRIQUE DE MODÈLES ====================
# Le tokenizer est construit UNE fois. Tous les paliers partagent donc exactement
# les mêmes ids de tokens, la même taille d'embedding et la même taille de softmax.
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from cellid_encoding import HOUR_VOCAB
from user_groups import group_vocab

set_seed(CONFIG["seed"])

# Un modèle >= 7B ne tient pas en local : GPU obligatoire, chargé en 4-bit (QLoRA)
BIG_MODEL = any(t in CONFIG["model_name"] for t in ("7B", "8B", "13B", "70B"))
if BIG_MODEL and DEVICE != "cuda":
    raise RuntimeError(f"{CONFIG['model_name']} nécessite un GPU (Colab).")

USE_GROUPS = bool(CONFIG.get("use_groups")) and HAS_HOURS
if bool(CONFIG.get("use_groups")) and not HAS_HOURS:
    print("⚠️  use_groups=True mais le jeu n'a pas d'heures → cross-training par groupe "
          "désactivé (les groupes sont définis par tranche horaire).")
N_GROUPS = CONFIG["n_groups"] if USE_GROUPS else 0
GROUP_VOCAB = group_vocab(N_GROUPS) if USE_GROUPS else []

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:          # Mistral n'a pas de pad token
    tokenizer.pad_token = tokenizer.eos_token
print(f"{tokenizer.add_tokens(VOCAB)} tokens cell_id ajoutés au tokenizer")
if HAS_HOURS:
    print(f"{tokenizer.add_tokens(HOUR_VOCAB)} tokens heure ajoutés")
if USE_GROUPS:
    print(f"{tokenizer.add_tokens(GROUP_VOCAB)} tokens de groupe ajoutés ({', '.join(GROUP_VOCAB)})")

CELL_TOKEN_IDS = tokenizer.convert_tokens_to_ids(VOCAB)
HOUR_TOKEN_IDS = tokenizer.convert_tokens_to_ids(HOUR_VOCAB) if HAS_HOURS else []
GROUP_TOKEN_IDS = tokenizer.convert_tokens_to_ids(GROUP_VOCAB) if USE_GROUPS else []
NEW_TOKEN_IDS = CELL_TOKEN_IDS + HOUR_TOKEN_IDS + GROUP_TOKEN_IDS

PREFIX_IDS = tokenizer(CONFIG["prefix_text"], add_special_tokens=False).input_ids
CELL_TO_ID = {c: i for c, i in zip(VOCAB, CELL_TOKEN_IDS)}
HOUR_TO_ID = {h: i for h, i in zip(HOUR_VOCAB, HOUR_TOKEN_IDS)} if HAS_HOURS else {}
GROUP_TO_ID = {g: i for g, i in zip(GROUP_VOCAB, GROUP_TOKEN_IDS)} if USE_GROUPS else {}

CELL_IDS_T = torch.tensor(CELL_TOKEN_IDS)
CELL_POS = {tid: i for i, tid in enumerate(CELL_TOKEN_IDS)}   # id de token -> indice dans VOCAB


def new_model():
    """Un modèle de base NEUF, enveloppé dans un adaptateur LoRA NEUF.

    Recharger la base à chaque palier plutôt que de désenvelopper l'adaptateur du
    précédent : `get_peft_model` modifie le modèle EN PLACE, donc le rappeler sur un
    modèle déjà enveloppé empile les adaptateurs et fait démarrer le palier N sur
    les poids du palier N-1 — ce qui transformerait la courbe de scaling en courbe
    d'entraînement continué, exactement ce qu'on ne mesure pas ici.

    `trainable_token_indices` fait vivre les embeddings des nouveaux tokens DANS
    l'adaptateur PEFT : ils repartent eux aussi de zéro à chaque palier.
    """
    set_seed(CONFIG["seed"])   # même init LoRA partout : l'écart mesuré vient des
                               # données, pas du tirage d'initialisation
    if BIG_MODEL:
        from transformers import BitsAndBytesConfig
        from peft import prepare_model_for_kbit_training
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"],
                                                     quantization_config=bnb, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=DTYPE)

    # Redimensionnement AVANT prepare_model_for_kbit_training, init simple
    # (mean_resizing provoque des erreurs CUDA sur modèles quantifiés/fp16)
    model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
    if BIG_MODEL:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False

    # Si les embeddings d'entrée et de sortie ne sont PAS liés (cas de Mistral), il
    # faut aussi entraîner les lignes du lm_head, sinon les logits des nouveaux
    # tokens restent figés à leur init aléatoire.
    tied = getattr(model.config, "tie_word_embeddings", False)
    trainable_tokens = ({"embed_tokens": NEW_TOKEN_IDS} if tied
                        else {"embed_tokens": NEW_TOKEN_IDS, "lm_head": NEW_TOKEN_IDS})
    lora_cfg = LoraConfig(
        task_type="CAUSAL_LM",
        r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        trainable_token_indices=trainable_tokens,
    )
    model = get_peft_model(model, lora_cfg)
    if not BIG_MODEL:
        model.to(DEVICE)
    return model

# Premier chargement : télécharge les poids si besoin, pour qu'aucun palier ne paie
# le téléchargement au milieu de la boucle. Le modèle est libéré aussitôt.
import gc
_warm = new_model()
_warm.print_trainable_parameters()
del _warm
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("Poids du modèle en cache local ✅ — chaque palier rechargera depuis le disque.")


In [ ]:
# ==================== CONTEXTE D'UN PALIER ====================
# Tout ce qui dépend du palier vit ici, et NULLE PART ailleurs : les fonctions
# d'évaluation reçoivent leur `ctx` en argument, donc il est impossible d'évaluer
# le palier 10 000 avec les groupes du palier 5 000 sans s'en apercevoir.
import random
from cellid_encoding import encode_events, in_transition_window

if USE_GROUPS:
    from user_groups import (fit_groups, derive_group_windows, describe_groups,
                             group_token)


class TierContext:
    """État d'un palier : utilisateurs, groupes, créneaux, cellules rares, baselines."""

    def __init__(self, n_users, verbose=True):
        self.n_users = min(n_users, len(ALL_TRAIN))
        self.users_train = ALL_TRAIN[:self.n_users]
        self.users_test = TEST_USERS          # LE MÊME pour tous les paliers

        # --- fréquence des cell_id DANS CE PALIER -> cellules rares ----------
        # Un cell_id vu 3 fois dans le train n'a presque pas entraîné son embedding.
        # Séparer ces positions du reste, c'est isoler la part du gain d'échelle qui
        # vient de la couverture (voir des cellules assez souvent) de celle qui vient
        # d'une meilleure estimation des transitions déjà couvertes.
        self.cell_freq = Counter(c for _, cells, _ in self.users_train for c in cells)
        self.rare_cells = {c for c in VOCAB
                           if self.cell_freq.get(c, 0) < SCALE_CONFIG["rare_cell_threshold"]}

        # --- groupes ajustés sur le TRAIN DE CE PALIER uniquement ------------
        # Les utilisateurs de test n'influencent jamais la géométrie des groupes, et
        # un palier n'hérite jamais des centroïdes d'un autre : un groupe doit rester
        # dérivable des seules données disponibles à ce palier.
        if USE_GROUPS:
            self.group_model = fit_groups(
                self.users_train, n_groups=N_GROUPS,
                bands=[tuple(b) for b in CONFIG["group_bands"]],
                shrinkage=CONFIG["group_shrinkage"], seed=CONFIG["seed"])
            self.oracle_train = self.group_model.assign_all(self.users_train)
            self.oracle_test = self.group_model.assign_all(self.users_test)
            if CONFIG.get("group_windows"):
                self.group_windows = {int(g): [tuple(w) for w in ws]
                                      for g, ws in CONFIG["group_windows"].items()}
            else:
                self.group_windows = derive_group_windows(
                    self.users_train, self.oracle_train, N_GROUPS,
                    min_deficit=CONFIG["group_window_min_deficit"],
                    weight_gain=CONFIG["group_window_weight_gain"],
                    max_weight=CONFIG["group_window_max_weight"])
        else:
            self.group_model = None
            self.group_windows = {}
            self.oracle_train = [None] * len(self.users_train)
            self.oracle_test = [None] * len(self.users_test)

        # Étiquettes utilisées à l'ENTRAÎNEMENT : détectées sur le préfixe de
        # contexte, donc même distribution qu'à l'évaluation.
        self.train_group = [self.detect_group_prefix(cells, hours, CONFIG["context_fraction"])
                            for _, cells, hours in self.users_train]

        # Contexte d'entraînement : chaque utilisateur contribue son préfixe, le
        # reste (jamais entraîné) sert de cible de validation.
        self.context_users = []
        for (uid, cells, hours), grp in zip(self.users_train, self.train_group):
            cut = split_index(len(cells), CONFIG["context_fraction"])
            self.context_users.append(
                (uid, cells[:cut], hours[:cut] if hours is not None else None, grp))

        # --- validation : LES MÊMES UTILISATEURS À TOUS LES PALIERS ----------
        # Les paliers étant emboîtés, la tranche initiale du fichier appartient au
        # train de chacun. Valider sur « tout le train » ferait porter la décision
        # d'arrêt sur 400 utilisateurs à un palier et 21 000 à l'autre : deux règles
        # d'arrêt différentes, et une comparaison faussée par la seule variance.
        k = SCALE_CONFIG["val_subsample"]
        self.val_users = self.users_train if (k is None or k >= self.n_users) \
            else self.users_train[:k]

        self.baseline_val = baseline_persistence(self.val_users, CONFIG["context_fraction"])
        self.baseline_test = BASELINE_TEST
        if verbose:
            self.report()

    # ------------------------------------------------------------ groupes
    def detect_group(self, cells, hours):
        """Groupe d'un utilisateur à partir de la tranche fournie. À l'évaluation on
        ne lui passe QUE le préfixe de contexte : l'étiquette ne peut donc pas venir
        des positions que le modèle doit prédire."""
        if not USE_GROUPS:
            return None
        return self.group_model.assign(cells, hours) if hours is not None else 0

    def detect_group_prefix(self, cells, hours, fraction):
        cut = split_index(len(cells), fraction)
        return self.detect_group(cells[:cut], hours[:cut] if hours is not None else None)

    def windows_of_group(self, group):
        if not USE_GROUPS or group is None:
            return CONFIG["transition_windows"]
        return self.group_windows.get(group, [])

    # ------------------------------------------------------------ encodage
    def group_prefix_ids(self, group):
        if not USE_GROUPS or group is None:
            return PREFIX_IDS
        return PREFIX_IDS + [GROUP_TO_ID[group_token(group)]]

    def encode_sequence(self, cells, hours, n_masked_cells=0, group=None):
        """prefix (+ token de groupe) + (heure, cell_id) interleavés ; labels -100 sur
        le préfixe, les heures et les n_masked_cells premiers cell_id."""
        ids, labels, weights = encode_events(
            cells, hours, CELL_TO_ID, HOUR_TO_ID, self.group_prefix_ids(group),
            self.windows_of_group(group), n_masked_cells=n_masked_cells)
        L = CONFIG["max_seq_len"]
        ids, labels, weights = ids[:L], labels[:L], weights[:L]
        return {"input_ids": ids, "labels": labels, "weights": weights,
                "attention_mask": [1] * len(ids)}

    # ------------------------------------------------------------ rapport
    def group_rows(self):
        """Lignes de synthèse par groupe, sérialisables. `describe_groups` ne renvoie
        que {"group", "n"} pour un groupe VIDE, d'où les .get()."""
        if not USE_GROUPS:
            return []
        rows = describe_groups(self.users_train, self.oracle_train, self.group_model, N_GROUPS)
        rows_te = describe_groups(self.users_test, self.oracle_test, self.group_model, N_GROUPS)
        nan = float("nan")
        out = []
        for r, rt in zip(rows, rows_te):
            out.append({
                "group": int(r["group"]),
                "n_train": int(r.get("n", 0)), "share_train": float(r.get("share", 0.0)),
                "n_test": int(rt.get("n", 0)), "share_test": float(rt.get("share", 0.0)),
                "profile": [float(v) for v in r.get("profile", [])],
                "persistence": float(r.get("persistence", nan)),
                "mean_len": float(r.get("mean_len", nan)),
                "windows": [[int(s), int(e), float(w)]
                            for s, e, w in self.windows_of_group(r["group"])],
            })
        return out

    def report(self):
        st = TIER_STATS.get(self.n_users, {})
        print(f"  {self.n_users} utilisateurs train | validation sur {len(self.val_users)} "
              f"(les mêmes à tous les paliers) | test {len(self.users_test)} "
              f"| baseline test {self.baseline_test:.1%}")
        print(f"  {len(self.rare_cells)}/{len(VOCAB)} cell_id vus moins de "
              f"{SCALE_CONFIG['rare_cell_threshold']} fois dans ce palier"
              + (f" | couverture du vocabulaire de test {st['test_coverage']:.1%}" if st else ""))
        if not USE_GROUPS:
            return
        band_names = [b[0] for b in self.group_model.bands]
        band_hdr = "  ".join(f"{n[:7]:>7}" for n in band_names)
        print(f"  grp  train         test  | {band_hdr} | persist. | créneaux (heure→poids)")
        for r in self.group_rows():
            if not r["n_train"]:
                print(f"  G{r['group']}   (aucun utilisateur d'entraînement)")
                continue
            prof = "  ".join(f"{v:6.0%} " for v in r["profile"])
            wins = ", ".join(f"{s:02d}-{e:02d}h→×{w:.1f}" for s, e, w in r["windows"]) or "aucun"
            print(f"  G{r['group']}  {r['n_train']:5d} ({r['share_train']:.0%})  "
                  f"{r['n_test']:4d} ({r['share_test']:.0%}) | {prof}| "
                  f"{r['persistence']:7.1%}  | {wins}")


class CellDataset(torch.utils.data.Dataset):
    """Fenêtres glissantes sur les préfixes de contexte, loss sur les seuls cell_id."""

    def __init__(self, ctx, users):
        self.items, self.sample_weights, self.groups = [], [], []
        for user in users:
            uid, cells, hours = user[0], user[1], user[2]
            grp = user[3] if len(user) > 3 else None
            wins = ctx.windows_of_group(grp)
            for start, end, n_ctx in make_chunks(len(cells), CONFIG["chunk_len"],
                                                 CONFIG["chunk_stride"]):
                chunk_hours = hours[start:end] if hours is not None else None
                self.items.append(ctx.encode_sequence(cells[start:end], chunk_hours,
                                                      n_masked_cells=n_ctx, group=grp))
                self.groups.append(grp)
                has_transition = False
                if chunk_hours is not None:
                    for idx in range(n_ctx, end - start):
                        if in_transition_window(chunk_hours[idx], wins):
                            has_transition = True
                            break
                self.sample_weights.append(
                    CONFIG.get("transition_chunk_oversample", 1.0) if has_transition else 1.0)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

    def approx_ram_mb(self):
        """Empreinte mémoire approximative (listes Python d'entiers). Au plus grand
        palier l'objet dépasse le Go : c'est le poste RAM hôte le plus lourd du
        notebook, et la raison pour laquelle un palier est libéré avant le suivant."""
        n_tok = sum(len(it["input_ids"]) for it in self.items)
        return n_tok * 4 * 36 / 1e6      # 4 listes, ~36 o par élément (objet + pointeur)


def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad] * (L - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([b["attention_mask"] + [0] * (L - len(b["attention_mask"])) for b in batch]),
        "labels":         torch.tensor([b["labels"] + [-100] * (L - len(b["labels"])) for b in batch]),
        "weights":        torch.tensor([b["weights"] + [1.0] * (L - len(b["weights"])) for b in batch]),
    }

print("TierContext / CellDataset prêts.")


In [ ]:
# ==================== ÉVALUATION BATCHÉE : accuracy top-k ====================
# Repris de train_cellid_week.ipynb (batching : un forward par batch de 8 au lieu
# d'un forward par utilisateur), plus deux ajouts propres à l'étude de scalabilité :
#   - `rare_cells` : accuracy séparée sur les cibles dont le cell_id est rare DANS
#     LE TRAIN DU PALIER. C'est là que la couverture doit payer en premier.
#   - `per_user` enrichi (longueur de contexte, groupe) : tous les découpages qui
#     suivent — par groupe, par longueur, bootstrap apparié — se calculent ensuite
#     en pandas, SANS un seul forward supplémentaire.
from cellid_encoding import event_token_positions


@torch.no_grad()
def evaluate_cell_accuracy(model, ctx, users, ks=(1, 3, 5), context_fraction=None,
                           transition_windows=None, group_mode="detected",
                           oracle_groups=None, batch_size=None, per_user=False,
                           rare_cells=None):
    """Teacher forcing : pour chaque position i>=1, prédit le cell i depuis le contexte
    0..i-1 (toujours le VRAI historique, heures comprises si disponibles).
    Si context_fraction est fourni, ne compte que les positions de la CIBLE.
    Deux variantes top-k : classique (argmax sur les cell_id) et top1_seen (argmax
    restreint aux cellules DÉJÀ VUES dans le préfixe de l'utilisateur).

    group_mode -- comment le token <Gk> est choisi :
      "detected" : détecté sur le SEUL préfixe de contexte (protocole d'inférence réel).
      "oracle"   : détecté sur la séquence complète (borne supérieure, pas un score
                   annonçable : mesure ce que coûte une erreur de détection).
      "none"     : aucun token de groupe.
      un entier k: force <Gk> pour tous (ablation « mauvais groupe »).
    """
    was_training = model.training
    model.eval()
    bs = batch_size or SCALE_CONFIG["eval_batch_size"]
    kmax = max(ks)
    rare = rare_cells if rare_cells is not None else set()
    cell_ids_dev = CELL_IDS_T.to(DEVICE)

    # --- 1) préparation : encodage + choix du groupe, par utilisateur --------
    prepared = []
    detect_agree = detect_n = 0
    for ui, (uid, cells, hours) in enumerate(users):
        if len(cells) < 2:
            continue
        min_j = split_index(len(cells), context_fraction) if context_fraction is not None else 1
        if not USE_GROUPS or group_mode == "none":
            grp = None
        elif isinstance(group_mode, int):
            grp = group_mode
        elif group_mode == "oracle":
            grp = (oracle_groups[ui] if oracle_groups is not None
                   else ctx.detect_group(cells, hours))
        else:                                    # "detected" -- préfixe uniquement
            cut = (min_j if context_fraction is not None
                   else split_index(len(cells), CONFIG.get("group_detect_fraction", 0.5)))
            grp = ctx.detect_group(cells[:cut], hours[:cut] if hours is not None else None)
            ref = (oracle_groups[ui] if oracle_groups is not None
                   else ctx.detect_group(cells, hours))
            detect_agree += grp == ref
            detect_n += 1
        wins = transition_windows if transition_windows is not None else ctx.windows_of_group(grp)
        enc = ctx.encode_sequence(cells, hours, group=grp)
        prepared.append({"uid": uid, "cells": cells, "hours": hours, "grp": grp, "wins": wins,
                         "ids": enc["input_ids"], "n_prefix": len(ctx.group_prefix_ids(grp)),
                         "min_j": min_j})

    hits = {k: 0 for k in ks}
    hits_seen = total = 0
    hits_seen_in = tot_in = hits_seen_out = tot_out = 0
    hits_rare = tot_rare = hits_common = tot_common = 0
    win_hits, win_tots = Counter(), Counter()
    grp_hits, grp_tots = Counter(), Counter()
    per_user_rows = []

    def score_one(p, logits):
        """Scoring d'UN utilisateur sur ses logits (identique au mode unitaire)."""
        nonlocal hits_seen, total, hits_seen_in, tot_in, hits_seen_out, tot_out
        nonlocal hits_rare, tot_rare, hits_common, tot_common
        cells, hours, wins = p["cells"], p["hours"], p["wins"]
        positions = event_token_positions(len(cells), has_hours=hours is not None)
        seen = torch.zeros(len(CELL_TOKEN_IDS), dtype=torch.bool)
        u_hits = {k: 0 for k in ks}
        u_seen = u_tot = 0
        for j in range(len(cells)):
            token_pos = p["n_prefix"] + positions[j]
            if token_pos >= len(p["ids"]):
                break   # séquence tronquée par max_seq_len : rien de plus à évaluer
            target_pos = CELL_POS[CELL_TO_ID[cells[j]]]
            if j == 0:
                seen[target_pos] = True
                continue
            if j >= p["min_j"]:
                scores = logits[token_pos - 1]
                top = scores.topk(kmax).indices.tolist()
                for k in ks:
                    hit = int(target_pos in top[:k])
                    hits[k] += hit
                    u_hits[k] += hit
                masked = scores.clone()
                masked[~seen] = float("-inf")
                hit_seen = int(int(masked.argmax()) == target_pos)
                hits_seen += hit_seen
                u_seen += hit_seen
                total += 1
                u_tot += 1
                if cells[j] in rare:
                    tot_rare += 1; hits_rare += hit_seen
                else:
                    tot_common += 1; hits_common += hit_seen
                if p["grp"] is not None:
                    grp_tots[p["grp"]] += 1
                    grp_hits[p["grp"]] += hit_seen
                if wins and hours is not None:
                    h = hours[j]
                    in_any = False
                    for (start, end, _) in wins:
                        if start <= h < end:
                            win_tots[(start, end)] += 1
                            win_hits[(start, end)] += hit_seen
                            in_any = True
                            break
                    if in_any:
                        tot_in += 1; hits_seen_in += hit_seen
                    else:
                        tot_out += 1; hits_seen_out += hit_seen
            seen[target_pos] = True
        if per_user and u_tot:
            per_user_rows.append({"user_id": p["uid"], "group": p["grp"], "n": u_tot,
                                  "n_context": p["min_j"], "n_events": len(p["cells"]),
                                  "top1": u_hits[1] / u_tot, "top1_seen": u_seen / u_tot})

    # --- 2) forwards batchés, du plus long au plus court --------------------
    order = sorted(range(len(prepared)), key=lambda i: -len(prepared[i]["ids"]))
    pad = tokenizer.pad_token_id
    for b0 in range(0, len(order), bs):
        chunk = [prepared[i] for i in order[b0:b0 + bs]]
        L = max(len(p["ids"]) for p in chunk)
        input_ids = torch.tensor([p["ids"] + [pad] * (L - len(p["ids"])) for p in chunk],
                                 device=DEVICE)
        attn = torch.tensor([[1] * len(p["ids"]) + [0] * (L - len(p["ids"])) for p in chunk],
                            device=DEVICE)
        out = model(input_ids=input_ids, attention_mask=attn).logits
        # On ne garde que les colonnes des cell_id (369 sur ~152 000) et on repasse
        # sur CPU immédiatement : garder les logits complets saturerait la VRAM.
        sel = out[:, :, cell_ids_dev].float().cpu()
        del out
        for row, p in enumerate(chunk):
            score_one(p, sel[row, :len(p["ids"])])
        del sel

    if was_training:
        model.train()

    nan = float("nan")
    if total == 0:
        result = {**{f"top{k}": nan for k in ks}, "top1_seen": nan, "n": 0}
    else:
        result = {**{f"top{k}": hits[k] / total for k in ks},
                  "top1_seen": hits_seen / total, "n": total}
    result["top1_seen_in_window"] = hits_seen_in / tot_in if tot_in else nan
    result["top1_seen_out_window"] = hits_seen_out / tot_out if tot_out else nan
    result["n_in_window"] = tot_in
    result["top1_seen_rare"] = hits_rare / tot_rare if tot_rare else nan
    result["top1_seen_common"] = hits_common / tot_common if tot_common else nan
    result["n_rare"] = tot_rare
    result["window_breakdown"] = {w: {"acc": win_hits[w] / win_tots[w], "n": win_tots[w]}
                                  for w in sorted(win_tots) if win_tots[w]}
    result["group_breakdown"] = {g: {"acc": grp_hits[g] / grp_tots[g], "n": grp_tots[g]}
                                 for g in sorted(grp_tots) if grp_tots[g]}
    result["group_detect_agreement"] = (detect_agree / detect_n) if detect_n else nan
    if per_user:
        result["per_user"] = per_user_rows
    return result

print(f"Évaluation batchée prête (batch={SCALE_CONFIG['eval_batch_size']}, "
      f"validation sur {SCALE_CONFIG['val_subsample'] or 'tous les'} utilisateurs).")


In [ ]:
# ==================== TRAINER PONDÉRÉ + CALLBACKS ====================
import time
from transformers import Trainer, TrainingArguments, TrainerCallback
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler


class WeightedLossTrainer(Trainer):
    """Trainer standard, sauf que la cross-entropy est pondérée par position : les
    cell_id dont l'heure tombe dans un créneau de transition comptent plus fort.
    Sans heures (poids == 1.0 partout), équivalent à la loss standard."""

    def __init__(self, *args, sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_sampler = sampler

    def _get_train_sampler(self, *args, **kwargs):
        # *args : la signature du Trainer amont a changé selon la version de
        # transformers (train_dataset optionnel) — on l'absorbe pour rester compatible.
        if self.custom_sampler is not None:
            return self.custom_sampler
        return super()._get_train_sampler(*args, **kwargs)

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self._get_train_sampler(),
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        shift_weights = weights[..., 1:].contiguous()
        losses = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1),
            ignore_index=-100, reduction="none")
        mask = (shift_labels.view(-1) != -100).float()
        weighted = losses * shift_weights.reshape(-1) * mask
        denom = num_items_in_batch if num_items_in_batch is not None else mask.sum().clamp(min=1)
        return ((weighted.sum() / denom, outputs) if return_outputs else weighted.sum() / denom)


class RamGuardCallback(TrainerCallback):
    """Surveille la RAM système ; si elle devient critique : arrêt PROPRE du palier."""

    def __init__(self):
        self.triggered = False

    def on_step_end(self, args, state, control, **kw):
        if state.global_step % CONFIG["ram_check_every_steps"] != 0:
            return
        if DEVICE == "mps":
            torch.mps.empty_cache()
        vm = psutil.virtual_memory()
        if (vm.available / 1e9 < CONFIG["min_free_ram_gb"]
                or vm.percent / 100 > CONFIG["max_ram_used_fraction"]):
            print(f"\n⚠️  GUARD RAM : {vm.percent:.0f}% utilisée, "
                  f"{vm.available / 1e9:.1f} Go libres → arrêt propre de ce palier.")
            self.triggered = True
            control.should_save = True
            control.should_training_stop = True


class AccuracyStopCallback(TrainerCallback):
    """Évalue l'accuracy de validation, garde les meilleurs poids en RAM, stoppe sur
    plateau.

    La cadence dépend du protocole, et c'est le point délicat de ce notebook :
      - iso_epochs : une validation par epoch, comme les notebooks précédents.
      - iso_steps  : une validation tous les `eval_every_steps` PAS. Une epoch n'y
        veut plus rien dire — elle vaut 67 pas au palier 500 et 2 815 au palier
        21 000 —, donc s'arrêter « après 4 epochs sans progrès » serait une règle
        d'arrêt 42 fois plus permissive au grand palier qu'au petit.
    """

    def __init__(self, model, ctx, protocol):
        self.model, self.ctx, self.protocol = model, ctx, protocol
        self.history = []
        self.best = 0.0
        self.best_params = None
        self.best_at = None
        self.since_best = 0
        self.reason = None
        self._t0 = time.time()
        self.eval_sec_total = 0.0

    def _evaluate(self, state, control):
        t0 = time.time()
        m = evaluate_cell_accuracy(self.model, self.ctx, self.ctx.val_users,
                                   context_fraction=CONFIG["context_fraction"])
        dt = time.time() - t0
        self.eval_sec_total += dt
        score = m["top1_seen"]       # métrique pilote : argmax restreint au répertoire vu
        self.history.append({"step": int(state.global_step), "epoch": round(state.epoch or 0, 3),
                             "top1": m["top1"], "top1_seen": score, "top3": m["top3"],
                             "top5": m["top5"], "eval_sec": dt,
                             "elapsed_sec": time.time() - self._t0})
        print(f"    pas {state.global_step:5d} (ep {state.epoch or 0:5.2f}) | "
              f"val top1={m['top1']:.1%}  top1_seen={score:.1%}  top3={m['top3']:.1%}  "
              f"top5={m['top5']:.1%}  ({dt:.0f}s d'éval)")
        if score > self.best:
            self.best, self.since_best = score, 0
            self.best_at = int(state.global_step)
            # Copie CPU des poids entraînables : l'éval finale utilise le MEILLEUR
            # modèle, pas celui (sur-appris) du dernier pas.
            self.best_params = {n: p.detach().cpu().clone()
                                for n, p in self.model.named_parameters() if p.requires_grad}
        else:
            self.since_best += 1
        if DEVICE == "mps":
            torch.mps.empty_cache()
        patience = (SCALE_CONFIG["early_stop_evals"] if self.protocol == "iso_steps"
                    else CONFIG["early_stop_patience"])
        if self.since_best >= patience:
            unit = "validations" if self.protocol == "iso_steps" else "epochs"
            self.reason = (f"plateau : {patience} {unit} sans progrès "
                           f"(meilleur = {self.best:.1%})")
            control.should_training_stop = True

    def on_step_end(self, args, state, control, **kw):
        if self.protocol != "iso_steps":
            return
        every = SCALE_CONFIG["eval_every_steps"]
        if state.global_step > 0 and state.global_step % every == 0:
            self._evaluate(state, control)

    def on_epoch_end(self, args, state, control, **kw):
        if self.protocol == "iso_steps":
            return
        self._evaluate(state, control)


def batch_sizes():
    """Batch d'entraînement. IDENTIQUE aux notebooks précédents : monter le batch
    pour aller plus vite serait une fausse bonne idée — à learning rate constant, un
    batch 4x plus grand donne 4x moins de pas d'optimisation, et le modèle apprend
    autre chose, pas la même chose plus vite."""
    if SCALE_CONFIG["train_batch_size"]:
        return SCALE_CONFIG["train_batch_size"], SCALE_CONFIG["grad_accum"] or 1
    if DEVICE == "cuda":
        return (4, 2) if BIG_MODEL else (8, 1)
    if DEVICE == "mps":
        return 4, 2
    return 1, 8


def training_arguments(ckpt_dir, protocol):
    """Arguments d'entraînement. Les DEUX protocoles partagent learning rate, warmup,
    batch, LoRA et seed ; ils ne diffèrent que par l'horizon :

      iso_steps  : `max_steps` fixe -> le scheduler cosine décroît sur EXACTEMENT le
                   même horizon à tous les paliers. C'est ce qui rend les paliers
                   comparables à compute constant.
      iso_epochs : `num_train_epochs` fixe -> horizon proportionnel à la taille du
                   jeu, comme dans train_cellid_llm.ipynb. Comparable aux runs
                   précédents, mais le coût croît linéairement avec le palier.
    """
    batch_size, grad_accum = batch_sizes()
    common = dict(
        output_dir=str(ckpt_dir),
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=CONFIG["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=CONFIG["warmup_steps"],
        logging_steps=50,
        weight_decay=CONFIG["weight_decay"],
        # Aucun checkpoint périodique : les meilleurs poids vivent en RAM.
        save_strategy="no",
        save_total_limit=CONFIG["save_total_limit"],
        bf16=(DEVICE == "cuda" and DTYPE == torch.bfloat16),
        fp16=(DEVICE == "cuda" and DTYPE == torch.float16),
        report_to=[],
        seed=CONFIG["seed"],
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        disable_tqdm=False,
    )
    if protocol == "iso_steps":
        # num_train_epochs est ignoré quand max_steps est fixé, mais on met une valeur
        # assez grande pour que le dataloader puisse boucler autant que nécessaire au
        # petit palier (1500 pas = 22 passages sur 534 fenêtres).
        return TrainingArguments(max_steps=SCALE_CONFIG["max_steps"],
                                 num_train_epochs=10_000, **common)
    return TrainingArguments(num_train_epochs=CONFIG["max_epochs"], **common)

print("Trainer et callbacks prêts. Protocoles : "
      + ", ".join(sorted({p for p, _ in JOBS})))


In [ ]:
# ==================== CALIBRATION : COMBIEN DE TEMPS ÇA VA PRENDRE ? ====================
# Le pire scénario de ce notebook n'est pas qu'il plante : c'est qu'il tourne quatre
# heures pour mourir sur la limite de session avant le dernier palier. On mesure donc
# le coût réel d'un pas et d'une évaluation AVANT de lancer le plan, et on projette.
import gc

def calibrate(n_steps=None, tier=None):
    """Mesure sec/pas d'entraînement et sec/évaluation, sur le plus petit palier."""
    n_steps = n_steps or SCALE_CONFIG["calibration_steps"]
    tier = tier or min(TIERS)
    print(f"Calibration sur le palier {tier} ({n_steps} pas + 1 validation + 1 éval de test)…")
    ctx = TierContext(tier, verbose=False)
    model = new_model()
    ds = CellDataset(ctx, ctx.context_users)
    bs, accum = batch_sizes()

    args = TrainingArguments(
        output_dir=str(Path(CONFIG["output_dir"]) / "calib"), max_steps=n_steps,
        per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
        learning_rate=CONFIG["learning_rate"], warmup_steps=1, logging_steps=10_000,
        save_strategy="no", report_to=[], seed=CONFIG["seed"],
        bf16=(DEVICE == "cuda" and DTYPE == torch.bfloat16),
        fp16=(DEVICE == "cuda" and DTYPE == torch.float16),
        dataloader_pin_memory=False, remove_unused_columns=False, disable_tqdm=True)
    trainer = WeightedLossTrainer(model=model, args=args, train_dataset=ds,
                                  data_collator=collate)
    t0 = time.time()
    trainer.train()
    sec_per_step = (time.time() - t0) / n_steps

    t0 = time.time()
    evaluate_cell_accuracy(model, ctx, ctx.val_users, context_fraction=CONFIG["context_fraction"])
    sec_val = time.time() - t0

    t0 = time.time()
    evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=CONFIG["context_fraction"])
    sec_test = time.time() - t0

    del trainer, model, ds, ctx
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()
    # Le premier pas paie la compilation/allocation : la mesure est donc un majorant.
    print(f"  {sec_per_step:.2f} s/pas | validation ({len(ALL_TRAIN[:SCALE_CONFIG['val_subsample'] or tier])} "
          f"utilisateurs) {sec_val:.0f} s | évaluation test {sec_test:.0f} s")
    return {"sec_per_step": sec_per_step, "sec_val": sec_val, "sec_test": sec_test}


def steps_per_epoch_of(n_windows):
    """Pas d'OPTIMISATION par epoch, en comptant comme le Trainer : ceil sur les
    batches, puis ceil sur l'accumulation de gradient. Un floor donnerait un
    `epochs_equiv` faux de quelques pour cent au petit palier."""
    import math
    bs, accum = batch_sizes()
    return max(1, math.ceil(math.ceil(n_windows / bs) / accum))


def project(cal):
    """Planning projeté, job par job. Pour iso_epochs le nombre d'epochs réellement
    exécutées dépend de l'early stopping : on encadre entre le minimum imposé par la
    patience et le plafond max_epochs."""
    n_diag = 13          # passes d'évaluation des diagnostics lourds
    n_base = 1 + len([f for f in SCALE_CONFIG["test_context_fractions"]
                      if f != CONFIG["context_fraction"]])   # fractions évaluées
    total_lo = total_hi = 0.0
    print(f"\n{'job':<18} {'pas':>12} {'entraîn.':>10} {'évals':>7} {'total':>14}")
    for protocol, tier in JOBS:
        tier = min(tier, len(ALL_TRAIN))
        spe = steps_per_epoch_of(TIER_STATS[tier]["n_windows"])
        n_eval_test = n_base + (n_diag if is_diagnostics_tier(protocol, tier) else 0)
        if protocol == "iso_steps":
            steps_lo = steps_hi = SCALE_CONFIG["max_steps"]
            n_val_lo = n_val_hi = steps_hi // SCALE_CONFIG["eval_every_steps"]
        else:
            # Plancher : la patience impose au moins patience+1 validations, donc
            # autant d'epochs -- sans jamais dépasser le plafond max_epochs.
            ep_lo = min(CONFIG["early_stop_patience"] + 1, CONFIG["max_epochs"])
            steps_lo, steps_hi = spe * ep_lo, spe * CONFIG["max_epochs"]
            n_val_lo, n_val_hi = ep_lo, CONFIG["max_epochs"]
        lo = steps_lo * cal["sec_per_step"] + n_val_lo * cal["sec_val"] + n_eval_test * cal["sec_test"]
        hi = steps_hi * cal["sec_per_step"] + n_val_hi * cal["sec_val"] + n_eval_test * cal["sec_test"]
        total_lo += lo
        total_hi += hi
        steps_txt = f"{steps_lo}" if steps_lo == steps_hi else f"{steps_lo}-{steps_hi}"
        span = (f"{hi / 60:.0f} min" if abs(hi - lo) < 60
                else f"{lo / 60:.0f}-{hi / 60:.0f} min")
        print(f"{job_name(protocol, tier):<18} {steps_txt:>12} "
              f"{steps_hi * cal['sec_per_step'] / 60:>9.0f}m {n_eval_test + n_val_hi:>7} {span:>14}")
    print(f"\nTOTAL projeté : {total_lo / 60:.0f} à {total_hi / 60:.0f} min "
          f"({total_lo / 3600:.1f}-{total_hi / 3600:.1f} h) pour les {len(JOBS)} jobs.")
    print("La borne haute suppose qu'aucun job iso_epochs ne s'arrête avant le plafond ;")
    print("la borne basse, qu'ils s'arrêtent tous au plus tôt. La mesure du premier palier")
    print("recalera l'ETA imprimée par la boucle.")
    budget = SCALE_CONFIG["time_budget_min"]
    if budget and total_hi / 60 > budget:
        print(f"\n⚠️  La projection haute ({total_hi / 60:.0f} min) dépasse le budget annoncé "
              f"({budget} min).")
        print("    Options, par ordre de perte d'information croissante :")
        print("      1. laisser tourner — la boucle est REPRENABLE, relancez-la après reconnexion ;")
        print("      2. retirer les paliers iso_epochs les plus gros (SCALE_CONFIG['runs']) ;")
        print("      3. mettre diagnostics_tiers=[] (économise ~13 passes d'éval par palier) ;")
        print("      4. réduire max_steps — mais alors TOUS les paliers doivent être refaits,")
        print("         un palier à 1500 pas et un autre à 800 ne sont pas sur la même courbe.")
    return {"total_lo_min": total_lo / 60, "total_hi_min": total_hi / 60, **cal}


def is_diagnostics_tier(protocol, tier):
    """Les diagnostics lourds (ablations, éval globale) tournent-ils pour ce job ?"""
    spec = SCALE_CONFIG["diagnostics_tiers"]
    if spec == "extremes":
        tiers = sorted({min(t, len(ALL_TRAIN)) for p, t in JOBS if p == protocol})
        return tier in {tiers[0], tiers[-1]}
    return tier in (spec or [])

CALIBRATION = project(calibrate()) if SCALE_CONFIG["calibrate"] else None


In [ ]:
# ==================== UN PALIER = UN ENTRAÎNEMENT COMPLET ====================

def _jsonable(obj):
    """Rend un résultat sérialisable : les clés tuple (4, 6) deviennent "04-06h",
    les scalaires numpy deviennent des float."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(k, tuple) and len(k) == 2:
                k = f"{int(k[0]):02d}-{int(k[1]):02d}h"
            out[str(k)] = _jsonable(v)
        return out
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (bool, str)) or obj is None:
        return obj
    if hasattr(obj, "item"):          # scalaires numpy / torch
        return obj.item()
    if isinstance(obj, int):
        return int(obj)
    if isinstance(obj, float):
        return float(obj)
    return obj


def sanity_check(model, ctx, train_ds):
    """Détecte AVANT l'entraînement les deux pannes classiques : des ids de tokens hors
    des tables d'embedding (crash CUDA « device-side assert », asynchrone donc
    illisible), et des labels négatifs autres que -100 (IndexError cryptique)."""
    n_embed = model.get_input_embeddings().weight.shape[0]
    out_emb = model.get_output_embeddings()
    n_out = out_emb.weight.shape[0] if out_emb is not None else n_embed
    max_id = max(max(CELL_TOKEN_IDS), max(PREFIX_IDS),
                 *(HOUR_TOKEN_IDS or [0]), *(GROUP_TOKEN_IDS or [0]))
    print(f"  tokenizer: {len(tokenizer)} | embed_tokens: {n_embed} | lm_head: {n_out} | "
          f"id max utilisé: {max_id}")
    assert max_id < n_embed and max_id < n_out, (
        "Ids de tokens hors des tables d'embedding : resize_token_embeddings n'a pas "
        "été appliqué. Relancez la cellule TOKENIZER.")
    bad = {l for item in (train_ds[0], train_ds[1]) for l in item["labels"] if l < 0 and l != -100}
    assert not bad, f"Labels invalides {bad} — seule la valeur -100 masque la loss."
    batch = collate([train_ds[0], train_ds[1]])
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "weights"}
    loss = model(**inputs).loss
    loss.backward()
    model.zero_grad()
    print(f"  forward/backward OK — loss initiale : {float(loss.detach()):.2f}")


def run_tier(protocol, tier, verbose=True):
    """Entraîne et évalue UN palier, de bout en bout. Rien n'est partagé avec les
    autres paliers hors le tokenizer, le vocabulaire et le jeu de test — qui sont
    communs par construction de l'expérience."""
    name = job_name(protocol, tier)
    tier = min(tier, len(ALL_TRAIN))
    t_start = time.time()
    print(f"\n{'=' * 78}\n  {name} — préparation ({tier} utilisateurs, protocole {protocol})\n{'=' * 78}")
    ctx = TierContext(tier, verbose=verbose)

    model = new_model()
    train_ds = CellDataset(ctx, ctx.context_users)
    n_touch = sum(1 for w in train_ds.sample_weights if w > 1.0)
    bs, accum = batch_sizes()
    steps_per_epoch = steps_per_epoch_of(len(train_ds))
    print(f"  {len(train_ds)} fenêtres (chunk={CONFIG['chunk_len']}, "
          f"stride={CONFIG['chunk_stride']}) ≈ {steps_per_epoch} pas/epoch, "
          f"~{train_ds.approx_ram_mb():.0f} Mo de RAM"
          + (f" — {n_touch} en créneau de transition" if HAS_HOURS else ""))
    sanity_check(model, ctx, train_ds)

    # --- entraînement -------------------------------------------------------
    ckpt_dir = Path(CONFIG["output_dir"]) / "checkpoints" / name
    sampler = None
    if HAS_HOURS and any(w > 1.0 for w in train_ds.sample_weights):
        sampler = WeightedRandomSampler(weights=train_ds.sample_weights,
                                        num_samples=len(train_ds), replacement=True)
    ram_cb = RamGuardCallback()
    acc_cb = AccuracyStopCallback(model, ctx, protocol)
    trainer = WeightedLossTrainer(model=model, args=training_arguments(ckpt_dir, protocol),
                                  train_dataset=train_ds, data_collator=collate,
                                  callbacks=[ram_cb, acc_cb], sampler=sampler)
    horizon = (f"{SCALE_CONFIG['max_steps']} pas" if protocol == "iso_steps"
               else f"{CONFIG['max_epochs']} epochs")
    print(f"\n  {name} — entraînement (horizon {horizon})")
    t_train = time.time()
    trainer.train()
    train_sec = time.time() - t_train
    steps_done = int(trainer.state.global_step)
    stop_reason = ("guard RAM déclenché" if ram_cb.triggered
                   else acc_cb.reason or "horizon atteint")
    print(f"  Entraînement terminé en {train_sec / 60:.1f} min "
          f"({steps_done} pas) — {stop_reason}")

    # On évalue le MEILLEUR modèle (pic de validation), pas celui du dernier pas.
    if acc_cb.best_params is not None:
        model.load_state_dict(acc_cb.best_params, strict=False)
        print(f"  Meilleur modèle rechargé (val top1_seen = {acc_cb.best:.1%} "
              f"au pas {acc_cb.best_at})")

    tokens_seen = steps_done * bs * accum * (CONFIG["chunk_len"] * (2 if HAS_HOURS else 1))
    record = {
        "job": name, "protocol": protocol, "n_users": tier, "status": "trained",
        "files": {k: str(v) for k, v in FILES.items()},
        "data": _jsonable(TIER_STATS[tier]),
        "config": {k: CONFIG[k] for k in ("model_name", "seed", "max_epochs",
                                          "early_stop_patience", "learning_rate", "lora_r",
                                          "lora_alpha", "chunk_len", "chunk_stride",
                                          "context_fraction", "use_groups", "n_groups")},
        "scale_config": {k: SCALE_CONFIG[k] for k in ("max_steps", "eval_every_steps",
                                                      "early_stop_evals", "val_subsample",
                                                      "rare_cell_threshold")},
        "vocab_size_shared": len(VOCAB),
        "has_hours": HAS_HOURS,
        "groups": ctx.group_rows(),
        "training": {"history": acc_cb.history, "best_val_top1_seen": acc_cb.best,
                     "best_step": acc_cb.best_at, "steps_run": steps_done,
                     "steps_per_epoch": steps_per_epoch,
                     "epochs_equiv": steps_done / steps_per_epoch,
                     "stop_reason": stop_reason, "train_sec": train_sec,
                     "eval_sec_during_training": acc_cb.eval_sec_total,
                     "baseline_val": ctx.baseline_val,
                     "n_train_windows": len(train_ds),
                     "tokens_seen_max": tokens_seen,
                     "batch_size": bs, "grad_accum": accum},
    }
    # Écriture intermédiaire : si la session tombe pendant les évaluations,
    # l'entraînement n'est pas perdu silencieusement.
    (RESULTS_DIR / f"{name}.json").write_text(
        json.dumps(_jsonable(record), ensure_ascii=False, indent=1), encoding="utf-8")

    # --- évaluation sur le test (les MÊMES utilisateurs à tous les paliers) --
    # Le balayage des fractions de contexte est le cœur de l'analyse « démarrage à
    # froid » : si l'échelle sert à quelque chose, c'est d'abord quand l'historique
    # du client est court.
    print(f"\n  {name} — évaluation")
    ref = CONFIG["context_fraction"]
    fracs = [ref] + [f for f in SCALE_CONFIG["test_context_fractions"] if f != ref]
    by_fraction = {}
    for frac in fracs:
        m = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=frac,
                                   group_mode="detected", oracle_groups=ctx.oracle_test,
                                   per_user=(frac == ref), rare_cells=ctx.rare_cells)
        base = baseline_persistence(ctx.users_test, frac)
        if frac == ref:
            record["per_user"] = m.pop("per_user")
        by_fraction[f"{frac:.2f}"] = {"metrics": _jsonable(m), "baseline": base,
                                      "delta": max(m["top1"], m["top1_seen"]) - base}
        tag = "  (référence)" if frac == ref else ""
        print(f"    contexte {frac:.0%}{tag:<13} | n={m['n']:5d} | top1={m['top1']:.1%}  "
              f"top1_seen={m['top1_seen']:.1%}  top3={m['top3']:.1%}  top5={m['top5']:.1%}  "
              f"| baseline {base:.1%}  écart {by_fraction[f'{frac:.2f}']['delta']:+.1%}")
    record["test"] = {"by_fraction": by_fraction, "reference_fraction": f"{ref:.2f}"}
    m_ref = by_fraction[f"{ref:.2f}"]["metrics"]
    print(f"    cellules rares (<{SCALE_CONFIG['rare_cell_threshold']} vues) : "
          f"{m_ref['top1_seen_rare']:.1%} sur n={m_ref['n_rare']}  |  "
          f"cellules fréquentes : {m_ref['top1_seen_common']:.1%}")

    if HAS_HOURS:
        def test_windows_of(ui):
            _, cells, hours = ctx.users_test[ui]
            return ctx.windows_of_group(ctx.detect_group_prefix(cells, hours, ref))
        bd = baseline_persistence_breakdown(ctx.users_test, ref, test_windows_of)
        record["baseline_breakdown"] = _jsonable(bd)
        print(f"    créneaux de transition : modèle {m_ref['top1_seen_in_window']:.1%} dedans / "
              f"{m_ref['top1_seen_out_window']:.1%} dehors  |  baseline "
              f"{bd['in_window']:.1%} / {bd['out_window']:.1%}")

    # --- diagnostics lourds : seulement aux paliers désignés ----------------
    # Chacun est une passe d'évaluation complète sur les 1000 utilisateurs de test.
    # Les faire partout doublerait le coût du notebook pour une information qui, elle,
    # n'a besoin d'être comparée qu'entre les extrêmes de l'échelle.
    if is_diagnostics_tier(protocol, tier):
        m_all = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=None,
                                       group_mode="detected", oracle_groups=ctx.oracle_test,
                                       rare_cells=ctx.rare_cells)
        base_all = baseline_persistence(ctx.users_test, context_fraction=None)
        record["global_eval"] = {"metrics": _jsonable(m_all), "baseline": base_all}
        print(f"    global (séquence complète)  | n={m_all['n']:5d} | "
              f"top1_seen={m_all['top1_seen']:.1%} | baseline {base_all:.1%}  (diagnostic)")

        if USE_GROUPS:
            variants = [("détecté (protocole réel)", dict(group_mode="detected",
                                                          oracle_groups=ctx.oracle_test)),
                        ("oracle (borne supérieure)", dict(group_mode="oracle",
                                                           oracle_groups=ctx.oracle_test)),
                        ("sans token de groupe", dict(group_mode="none"))]
            variants += [(f"forcé à G{g}", dict(group_mode=g)) for g in range(N_GROUPS)]
            rows, ref_score = [], None
            for label, kw in variants:
                m = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=ref,
                                           rare_cells=ctx.rare_cells, **kw)
                if ref_score is None:
                    ref_score = m["top1_seen"]
                rows.append({"variant": label, "top1_seen": m["top1_seen"],
                             "delta_vs_detected": m["top1_seen"] - ref_score,
                             "group_detect_agreement": m["group_detect_agreement"]})
            record["ablation_group"] = _jsonable(rows)
            print("    ablation groupe : " + "  ".join(
                f"{r['variant'].split()[0]}={r['top1_seen']:.1%}" for r in rows[:3]))

        if HAS_HOURS:
            def corrupt_hours(users, mode, seed=CONFIG["seed"]):
                """Mêmes cell_id, mêmes longueurs, seules les heures changent."""
                rng_local = random.Random(seed)
                out = []
                for uid, cells, hours in users:
                    if hours is None:
                        out.append((uid, cells, hours)); continue
                    if mode == "shuffle":
                        h = list(hours); rng_local.shuffle(h)
                    elif mode == "fixed":
                        h = [12] * len(hours)
                    elif mode == "shift12":
                        h = [(hh + 12) % 24 for hh in hours]
                    else:
                        raise ValueError(mode)
                    out.append((uid, cells, h))
                return out

            # Le groupe est gardé FIXE (oracle) d'une variante à l'autre : sinon
            # corrompre les heures changerait aussi le groupe détecté, et on ne
            # saurait plus ce qu'on mesure.
            hour_rows = []
            for label, short, variant in (
                    ("heures réelles", "réelles", ctx.users_test),
                    ("heures mélangées", "mélangées", corrupt_hours(ctx.users_test, "shuffle")),
                    ("heure fixe (12h)", "fixe", corrupt_hours(ctx.users_test, "fixed")),
                    ("heures décalées +12h", "décalées", corrupt_hours(ctx.users_test, "shift12"))):
                m = evaluate_cell_accuracy(model, ctx, variant, context_fraction=ref,
                                           group_mode="oracle", oracle_groups=ctx.oracle_test)
                hour_rows.append({"variant": label, "short": short, "top1_seen": m["top1_seen"],
                                  "in_window": m["top1_seen_in_window"],
                                  "out_window": m["top1_seen_out_window"]})
            record["ablation_hour"] = _jsonable(hour_rows)
            print("    ablation heure : " + "  ".join(
                f"{r['short']}={r['top1_seen']:.1%}" for r in hour_rows))

    # --- sauvegarde ---------------------------------------------------------
    if SCALE_CONFIG["save_adapters"]:
        adapter_dir = RESULTS_DIR / "adapters" / name
        model.save_pretrained(adapter_dir)
        record["adapter_dir"] = str(adapter_dir)

    record["status"] = "complete"
    record["runtime_sec"] = time.time() - t_start
    (RESULTS_DIR / f"{name}.json").write_text(
        json.dumps(_jsonable(record), ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"  ✅ {name} terminé en {record['runtime_sec'] / 60:.1f} min "
          f"→ {RESULTS_DIR / (name + '.json')}")

    # --- libération : sans ça, la VRAM du palier N-1 fait échouer le palier N
    del trainer, model, train_ds, ctx, acc_cb
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()
    return record

print("run_tier() prêt.")


In [ ]:
# ==================== BOUCLE SUR LE PLAN (REPRENABLE) ====================
# Cette cellule peut être relancée telle quelle après une déconnexion Colab : les
# jobs déjà terminés sont relus depuis leur JSON et sautés. Pour en refaire un,
# mettez son nom (ex. "iso_steps_05000") dans SCALE_CONFIG["force_rerun"].
# Les jobs sont exécutés du plus petit palier au plus grand : si la session tombe,
# c'est toujours le HAUT de la courbe qui manque, jamais le bas — donc ce qui est
# déjà calculé reste interprétable.

RESULTS = {}
t_all = time.time()
done, todo = [], []
for protocol, tier in JOBS:
    tier = min(tier, len(ALL_TRAIN))
    name = job_name(protocol, tier)
    path = RESULTS_DIR / f"{name}.json"
    if path.exists() and name not in SCALE_CONFIG["force_rerun"]:
        rec = json.loads(path.read_text(encoding="utf-8"))
        if rec.get("status") == "complete":
            RESULTS[name] = rec
            done.append(name)
            continue
    todo.append((protocol, tier))

if done:
    print(f"Déjà terminés, sautés : {', '.join(done)}")
print(f"À entraîner ({len(todo)}) : "
      + (", ".join(job_name(p, t) for p, t in todo) if todo else "aucun — le plan est complet")
      + "\n")

for i, (protocol, tier) in enumerate(todo, 1):
    print(f"\n########## {i}/{len(todo)} : {job_name(protocol, tier)} ##########")
    RESULTS[job_name(protocol, tier)] = run_tier(protocol, tier)
    elapsed = time.time() - t_all
    if i < len(todo):
        # ETA pondérée par le nombre de fenêtres restantes, pas par le nombre de jobs :
        # les paliers n'ont pas du tout le même coût, une moyenne simple mentirait.
        done_w = sum(TIER_STATS[min(t, len(ALL_TRAIN))]["n_windows"] for _, t in todo[:i])
        left_w = sum(TIER_STATS[min(t, len(ALL_TRAIN))]["n_windows"] for _, t in todo[i:])
        eta = elapsed / max(done_w, 1) * left_w
        print(f"\n  ⏱  {elapsed / 60:.0f} min écoulées, ~{eta / 60:.0f} min restantes "
              f"pour les {len(todo) - i} jobs suivants")

# Relecture depuis le disque : la synthèse travaille sur ce qui est RÉELLEMENT
# écrit, donc elle donne le même résultat sur une session neuve sans GPU.
RESULTS = {}
for protocol, tier in JOBS:
    path = RESULTS_DIR / f"{job_name(protocol, min(tier, len(ALL_TRAIN)))}.json"
    if path.exists():
        rec = json.loads(path.read_text(encoding="utf-8"))
        if rec.get("status") == "complete":
            RESULTS[rec["job"]] = rec
print(f"\n{len(RESULTS)}/{len(JOBS)} jobs disponibles : {', '.join(RESULTS)}")


In [ ]:
# ==================== SYNTHÈSE : LA COURBE DE SCALING ====================
# Cette cellule et les suivantes ne lisent QUE les JSON : elles tournent sur une
# session neuve, sans GPU, tant que RESULTS_DIR pointe sur le dossier de résultats.
import pandas as pd
import numpy as np

def load_results(results_dir=None):
    d = Path(results_dir) if results_dir else RESULTS_DIR
    out = {}
    for p in sorted(d.glob("*.json")):
        rec = json.loads(p.read_text(encoding="utf-8"))
        if rec.get("status") == "complete" and "n_users" in rec:
            out[rec["job"]] = rec
    return out

RESULTS = load_results()
if not RESULTS:
    raise RuntimeError(f"Aucun résultat dans {RESULTS_DIR} — lancez d'abord la cellule BOUCLE.")

REF_KEY = f"{CONFIG['context_fraction']:.2f}"
PROTOCOLS = sorted({r["protocol"] for r in RESULTS.values()})

rows = []
for name, r in RESULTS.items():
    ref = r["test"]["by_fraction"][REF_KEY]
    m, tr = ref["metrics"], r["training"]
    rows.append({
        "job": name, "protocole": r["protocol"], "utilisateurs": r["n_users"],
        "événements": r["data"]["n_events"], "fenêtres": r["data"]["n_windows"],
        "pas": tr["steps_run"], "epochs_eq": tr["epochs_equiv"],
        "val_top1_seen": tr["best_val_top1_seen"],
        "top1": m["top1"], "top3": m["top3"], "top5": m["top5"], "top1_seen": m["top1_seen"],
        "baseline": ref["baseline"], "écart": ref["delta"],
        "rares": m.get("top1_seen_rare", float("nan")),
        "fréquentes": m.get("top1_seen_common", float("nan")),
        "dans_créneau": m.get("top1_seen_in_window", float("nan")),
        "détect.": m.get("group_detect_agreement", float("nan")),
        "min_GPU": r["runtime_sec"] / 60,
        "arrêt": tr["stop_reason"],
    })
SUMMARY = pd.DataFrame(rows).sort_values(["protocole", "utilisateurs"]).set_index("job")

# --- intervalles de confiance : bootstrap SUR LES UTILISATEURS ---------------
# Les prédictions d'un même utilisateur sont corrélées (même domicile, même
# répertoire de cellules) : un intervalle binomial sur n=14 700 prédictions serait
# très optimiste. On rééchantillonne donc les UTILISATEURS, qui sont, eux,
# indépendants. Et comme le jeu de test est le même à tous les paliers, la
# comparaison entre deux paliers se fait en APPARIÉ : on rééchantillonne un seul
# jeu d'utilisateurs et on lit la différence sur ces mêmes utilisateurs, ce qui
# annule la variance due au tirage du test.
RNG = np.random.default_rng(CONFIG["seed"])
N_BOOT = 2000

PER_USER = {name: pd.DataFrame(r["per_user"]).set_index("user_id")
            for name, r in RESULTS.items() if r.get("per_user")}

def _weighted(hits, n):
    return hits.sum() / n.sum()

def boot_ci(name, metric="top1_seen", n_boot=N_BOOT, alpha=0.05):
    """IC de l'accuracy d'un job, par bootstrap sur les utilisateurs."""
    df = PER_USER[name]
    n = df["n"].to_numpy(float)
    hits = (df[metric] * df["n"]).to_numpy(float)
    idx = RNG.integers(0, len(df), size=(n_boot, len(df)))
    vals = hits[idx].sum(1) / n[idx].sum(1)
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))

def paired_boot(name_a, name_b, metric="top1_seen", n_boot=N_BOOT, alpha=0.05):
    """IC de la DIFFÉRENCE b - a, en appariant sur les utilisateurs communs."""
    a, b = PER_USER[name_a], PER_USER[name_b]
    common = a.index.intersection(b.index)
    a, b = a.loc[common], b.loc[common]
    na, nb = a["n"].to_numpy(float), b["n"].to_numpy(float)
    ha = (a[metric] * a["n"]).to_numpy(float)
    hb = (b[metric] * b["n"]).to_numpy(float)
    idx = RNG.integers(0, len(common), size=(n_boot, len(common)))
    diffs = hb[idx].sum(1) / nb[idx].sum(1) - ha[idx].sum(1) / na[idx].sum(1)
    lo, hi = np.quantile(diffs, alpha / 2), np.quantile(diffs, 1 - alpha / 2)
    return {"delta": float(hb.sum() / nb.sum() - ha.sum() / na.sum()),
            "lo": float(lo), "hi": float(hi), "significatif": bool(lo > 0 or hi < 0),
            "n_users": int(len(common))}

for name in SUMMARY.index:
    if name in PER_USER:
        lo, hi = boot_ci(name)
        SUMMARY.loc[name, "IC_bas"] = lo
        SUMMARY.loc[name, "IC_haut"] = hi

pd.set_option("display.width", 220)
pct = ["val_top1_seen", "top1", "top3", "top5", "top1_seen", "baseline", "écart",
       "rares", "fréquentes", "dans_créneau", "détect.", "IC_bas", "IC_haut"]
wanted = ["protocole", "utilisateurs", "pas", "epochs_eq", "top1_seen", "IC_bas",
          "IC_haut", "baseline", "écart", "top1", "top3", "top5", "rares",
          "fréquentes", "min_GPU"]
show = SUMMARY[[c for c in wanted if c in SUMMARY.columns]].copy()
print(f"RÉFÉRENCE : contexte {CONFIG['context_fraction']:.0%}, {len(TEST_USERS)} utilisateurs "
      f"de test JAMAIS entraînés, identiques à tous les paliers.")
print(f"IC 95% par bootstrap sur les utilisateurs ({N_BOOT} tirages).\n")
print(show.to_string(formatters={**{c: "{:.1%}".format for c in pct if c in show.columns},
                                 "min_GPU": "{:.1f}".format, "epochs_eq": "{:.1f}".format}))

# --- la lecture qui compte : paliers consécutifs, en apparié ----------------
print("\n\nGAIN D'UN PALIER AU SUIVANT (bootstrap APPARIÉ sur les mêmes utilisateurs)")
print("Un gain est réel si son intervalle ne contient pas 0. C'est ce test, et non la")
print("comparaison des accuracies brutes, qui dit où la courbe sature.\n")
STEPS_TABLE = {}
for proto in PROTOCOLS:
    jobs = SUMMARY[SUMMARY["protocole"] == proto].sort_values("utilisateurs").index.tolist()
    jobs = [j for j in jobs if j in PER_USER]
    if len(jobs) < 2:
        continue
    print(f"  [{proto}]")
    rows_p = []
    for a, b in zip(jobs, jobs[1:]):
        res = paired_boot(a, b)
        na, nb = RESULTS[a]["n_users"], RESULTS[b]["n_users"]
        mark = "OUI" if res["significatif"] else "non"
        print(f"    {na:>6} → {nb:<6} : {res['delta']:+.2%}  "
              f"[{res['lo']:+.2%} ; {res['hi']:+.2%}]   réel : {mark}")
        rows_p.append({"de": na, "à": nb, **res})
    # Le gain total, du plus petit au plus grand palier
    tot = paired_boot(jobs[0], jobs[-1])
    print(f"    {RESULTS[jobs[0]]['n_users']:>6} → {RESULTS[jobs[-1]]['n_users']:<6} "
          f"(TOTAL) : {tot['delta']:+.2%}  [{tot['lo']:+.2%} ; {tot['hi']:+.2%}]")
    STEPS_TABLE[proto] = pd.DataFrame(rows_p)

# --- loi d'échelle ----------------------------------------------------------
# Ajustement accuracy = a + b * log10(N). Le coefficient b se lit directement :
# « chaque multiplication par 10 du nombre d'utilisateurs rapporte b points ».
print("\n\nLOI D'ÉCHELLE ajustée : top1_seen = a + b · log10(N_utilisateurs)")
SCALING_FIT = {}
for proto in PROTOCOLS:
    sub = SUMMARY[SUMMARY["protocole"] == proto].sort_values("utilisateurs")
    if len(sub) < 3:
        print(f"  [{proto}] moins de 3 paliers — pas d'ajustement.")
        continue
    x = np.log10(sub["utilisateurs"].to_numpy(float))
    y = sub["top1_seen"].to_numpy(float)
    b, a = np.polyfit(x, y, 1)
    pred = a + b * x
    ss_res = float(((y - pred) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
    SCALING_FIT[proto] = {"a": float(a), "b": float(b), "r2": r2}
    print(f"  [{proto}] b = {b:+.2%} par ×10 utilisateurs  (soit {b * np.log10(2):+.2%} "
          f"par doublement), R² = {r2:.2f}")
    # Extrapolation : à ce rythme, combien d'utilisateurs pour +1 point de plus ?
    if b > 0:
        need = 10 ** ((y[-1] + 0.01 - a) / b)
        print(f"           extrapolation (à prendre avec des pincettes hors du domaine "
              f"mesuré) : {need:,.0f} utilisateurs pour +1 point de plus.")

SUMMARY.to_csv(RESULTS_DIR / "scaling_summary.csv")
for proto, df in STEPS_TABLE.items():
    df.to_csv(RESULTS_DIR / f"scaling_gains_{proto}.csv", index=False)
print(f"\nCSV écrits dans {RESULTS_DIR}")


In [ ]:
# ==================== DÉTAILS : QUI PROFITE DE L'ÉCHELLE ? ====================
# Un score global qui monte de 2 points ne dit pas d'où viennent ces 2 points. Les
# quatre découpages ci-dessous répondent à « qui », « quand » et « sur quoi » — et
# tous se calculent depuis les lignes par utilisateur déjà stockées : aucun forward
# supplémentaire, aucun GPU.

ORDER = SUMMARY.sort_values(["protocole", "utilisateurs"]).index.tolist()

def _tier_label(name):
    return f"{RESULTS[name]['n_users']}"

# --- 1. par groupe de comportement ------------------------------------------
# Agrégé depuis les lignes par utilisateur, pondéré par le nombre de prédictions —
# strictement équivalent à une évaluation par groupe, mais gratuit.
if USE_GROUPS and PER_USER:
    print("ACCURACY PAR GROUPE DE COMPORTEMENT (top1_seen, contexte de référence)")
    print("L'hypothèse à tester : les groupes MOBILES (G2, G3), que la baseline prédit")
    print("mal, devraient profiter de l'échelle plus longtemps que les sédentaires (G0),")
    print("dont le comportement est déjà capturé par « répéter la dernière cellule ».\n")
    grp_rows = []
    for name in ORDER:
        if name not in PER_USER:
            continue
        df = PER_USER[name]
        row = {"job": name, "protocole": RESULTS[name]["protocol"],
               "utilisateurs": RESULTS[name]["n_users"]}
        for g, sub in df.groupby("group"):
            row[f"G{int(g)}"] = float((sub["top1_seen"] * sub["n"]).sum() / sub["n"].sum())
            row[f"G{int(g)} n"] = int(len(sub))
        grp_rows.append(row)
    GROUPS_DF = pd.DataFrame(grp_rows).set_index("job")
    cols = [c for c in GROUPS_DF.columns if c.startswith("G") and not c.endswith(" n")]
    print(GROUPS_DF[["utilisateurs"] + cols].to_string(
        formatters={c: "{:.1%}".format for c in cols}))
    GROUPS_DF.to_csv(RESULTS_DIR / "scaling_par_groupe.csv")
else:
    GROUPS_DF = pd.DataFrame()

# --- 2. démarrage à froid : accuracy selon la longueur du contexte -----------
# Le vrai argument produit de l'échelle. Un modèle nourri de 21 000 utilisateurs
# devrait surtout être meilleur QUAND L'HISTORIQUE DU CLIENT EST COURT : c'est là
# que la connaissance de population remplace la connaissance de l'individu.
print("\n\nDÉMARRAGE À FROID — top1_seen selon la fraction d'historique donnée en contexte")
print("(mêmes utilisateurs de test, seule la coupure contexte/cible change)\n")
cold_rows = []
for name in ORDER:
    r = RESULTS[name]
    row = {"job": name, "utilisateurs": r["n_users"]}
    for key, blk in sorted(r["test"]["by_fraction"].items()):
        row[f"ctx {float(key):.0%}"] = blk["metrics"]["top1_seen"]
        row[f"base {float(key):.0%}"] = blk["baseline"]
    cold_rows.append(row)
COLD_DF = pd.DataFrame(cold_rows).set_index("job")
cols_ctx = [c for c in COLD_DF.columns if c.startswith("ctx")]
cols_base = [c for c in COLD_DF.columns if c.startswith("base")]
print(COLD_DF[["utilisateurs"] + cols_ctx].to_string(
    formatters={c: "{:.1%}".format for c in cols_ctx}))
print("\nbaselines (identiques à tous les paliers) : " + "  ".join(
    f"{c.replace('base', 'ctx')}={COLD_DF[c].iloc[0]:.1%}" for c in cols_base))
if len(ORDER) >= 2:
    first, last = ORDER[0], ORDER[-1]
    gains = {c: COLD_DF.loc[last, c] - COLD_DF.loc[first, c] for c in cols_ctx}
    best_c = max(gains, key=gains.get)
    print(f"\nGain {RESULTS[first]['n_users']} → {RESULTS[last]['n_users']} utilisateurs, "
          f"par longueur de contexte : "
          + "  ".join(f"{c}={v:+.1%}" for c, v in gains.items()))
    print(f"→ l'échelle paie le plus à {best_c} ({gains[best_c]:+.1%}).")
COLD_DF.to_csv(RESULTS_DIR / "scaling_demarrage_a_froid.csv")

# --- 3. cellules rares -------------------------------------------------------
print("\n\nCELLULES RARES vs FRÉQUENTES (seuil : "
      f"{SCALE_CONFIG['rare_cell_threshold']} occurrences dans le train DU PALIER)")
print("Le seuil est relatif au palier : ce tableau montre à la fois que le nombre de")
print("cellules rares s'effondre avec N, et ce que le modèle fait des positions qui")
print("restent rares. Si la colonne « rares » reste plate pendant que « n rares »")
print("s'effondre, le gain d'échelle ne vient PAS de la couverture du vocabulaire.\n")
rare_rows = []
for name in ORDER:
    r = RESULTS[name]
    m = r["test"]["by_fraction"][REF_KEY]["metrics"]
    rare_rows.append({"job": name, "utilisateurs": r["n_users"],
                      "cell_id rares": r["data"]["n_rare"],
                      "couv. vocab test": r["data"]["test_coverage"],
                      "n prédictions rares": m.get("n_rare", 0),
                      "acc. rares": m.get("top1_seen_rare", float("nan")),
                      "acc. fréquentes": m.get("top1_seen_common", float("nan"))})
RARE_DF = pd.DataFrame(rare_rows).set_index("job")
print(RARE_DF.to_string(formatters={"acc. rares": "{:.1%}".format,
                                    "acc. fréquentes": "{:.1%}".format,
                                    "couv. vocab test": "{:.1%}".format}))
RARE_DF.to_csv(RESULTS_DIR / "scaling_cellules_rares.csv")

# --- 4. distribution par utilisateur ----------------------------------------
# Deux paliers de même moyenne peuvent cacher deux populations différentes : le
# décile bas dit si l'échelle relève les utilisateurs DIFFICILES ou seulement les
# faciles. C'est souvent la diapositive la plus parlante.
if PER_USER:
    print("\n\nDISTRIBUTION PAR UTILISATEUR (top1_seen, contexte de référence)\n")
    dist_rows = []
    for name in ORDER:
        if name not in PER_USER:
            continue
        df = PER_USER[name]
        q = df["top1_seen"].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
        dist_rows.append({"job": name, "utilisateurs": RESULTS[name]["n_users"],
                          "moyenne": float((df["top1_seen"] * df["n"]).sum() / df["n"].sum()),
                          "d1": q[0.1], "q1": q[0.25], "médiane": q[0.5], "q3": q[0.75],
                          "d9": q[0.9], "écart-type": df["top1_seen"].std()})
    DIST_DF = pd.DataFrame(dist_rows).set_index("job")
    print(DIST_DF.to_string(formatters={c: "{:.1%}".format for c in DIST_DF.columns
                                        if c != "utilisateurs"}))
    DIST_DF.to_csv(RESULTS_DIR / "scaling_distribution.csv")
else:
    DIST_DF = pd.DataFrame()

# --- 5. ablations (paliers de diagnostic seulement) -------------------------
ABL_GROUP_DF = pd.DataFrame([{"job": n, "utilisateurs": RESULTS[n]["n_users"],
                              **{v["variant"]: v["top1_seen"]
                                 for v in RESULTS[n]["ablation_group"]}}
                             for n in ORDER if RESULTS[n].get("ablation_group")])
if not ABL_GROUP_DF.empty:
    ABL_GROUP_DF = ABL_GROUP_DF.set_index("job")
    print("\n\nABLATION GROUPE (paliers de diagnostic) — le cross-training résiste-t-il à l'échelle ?")
    print("Hypothèse à vérifier : plus le train est gros, moins le token de groupe apporte,")
    print("parce que le modèle finit par déduire le profil de l'historique lui-même.\n")
    cols = [c for c in ABL_GROUP_DF.columns if c != "utilisateurs"]
    print(ABL_GROUP_DF.to_string(formatters={c: "{:.1%}".format for c in cols}))
    if "détecté (protocole réel)" in cols and "sans token de groupe" in cols:
        g = ABL_GROUP_DF["détecté (protocole réel)"] - ABL_GROUP_DF["sans token de groupe"]
        print("\nApport du token de groupe : " + "  ".join(
            f"{ABL_GROUP_DF.loc[i, 'utilisateurs']} users → {v:+.1%}" for i, v in g.items()))
    ABL_GROUP_DF.to_csv(RESULTS_DIR / "scaling_ablation_groupe.csv")

ABL_HOUR_DF = pd.DataFrame([{"job": n, "utilisateurs": RESULTS[n]["n_users"],
                             **{v["variant"]: v["top1_seen"]
                                for v in RESULTS[n]["ablation_hour"]}}
                            for n in ORDER if RESULTS[n].get("ablation_hour")])
if not ABL_HOUR_DF.empty:
    ABL_HOUR_DF = ABL_HOUR_DF.set_index("job")
    print("\n\nABLATION HEURE (paliers de diagnostic) — mêmes cell_id, seule l'heure change\n")
    cols = [c for c in ABL_HOUR_DF.columns if c != "utilisateurs"]
    print(ABL_HOUR_DF.to_string(formatters={c: "{:.1%}".format for c in cols}))
    ABL_HOUR_DF.to_csv(RESULTS_DIR / "scaling_ablation_heure.csv")

print(f"\n\nTous les CSV sont dans {RESULTS_DIR}")


In [ ]:
# ==================== GRAPHIQUES ====================
import matplotlib.pyplot as plt

# Palette validée (contraste, séparation daltonisme sur paires adjacentes). Les
# couleurs portent une IDENTITÉ (le protocole, le groupe) ou une MAGNITUDE (une
# seule série) — jamais un rang, jamais un dégradé arc-en-ciel.
SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_SOFT  = "#52514e"
INK_MUTED = "#8a8984"
GRID      = "#e6e5e1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
BLUE, ORANGE = SERIES[0], SERIES[1]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_SOFT, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "axes.edgecolor": GRID, "grid.color": GRID,
    "font.size": 10, "axes.titlesize": 12, "figure.dpi": 110,
})

def _clean(ax, ygrid=True):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if ygrid:
        ax.grid(axis="y", alpha=0.6, linewidth=0.8)
        ax.set_axisbelow(True)

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def _save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

COLOR_OF_PROTO = {p: SERIES[i % len(SERIES)] for i, p in enumerate(PROTOCOLS)}

# ------------------------------------------------- 1. LA courbe de scaling
# x en log : c'est l'échelle dans laquelle « ×2 utilisateurs » a la même largeur
# partout, donc la seule où une saturation se voit à l'œil.
fig, ax = plt.subplots(figsize=(9, 5))
for proto in PROTOCOLS:
    sub = SUMMARY[SUMMARY["protocole"] == proto].sort_values("utilisateurs")
    x = sub["utilisateurs"].to_numpy(float)
    y = sub["top1_seen"].to_numpy(float)
    if "IC_bas" in sub:
        err = np.vstack([y - sub["IC_bas"].to_numpy(float),
                         sub["IC_haut"].to_numpy(float) - y])
        ax.errorbar(x, y, yerr=err, color=COLOR_OF_PROTO[proto], linewidth=2,
                    marker="o", markersize=6, capsize=3, label=proto, zorder=3)
    else:
        ax.plot(x, y, color=COLOR_OF_PROTO[proto], linewidth=2, marker="o", label=proto)
    for xi, yi in zip(x, y):
        ax.annotate(f"{yi:.1%}", xy=(xi, yi), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=8.5, color=INK)
ax.axhline(BASELINE_TEST, color=INK_SOFT, linewidth=1.6, linestyle="--", zorder=2)
ax.annotate(f"baseline persistance {BASELINE_TEST:.1%}", xy=(0.01, BASELINE_TEST),
            xycoords=("axes fraction", "data"), xytext=(0, 5),
            textcoords="offset points", fontsize=9, color=INK_SOFT)
ax.set_xscale("log")
# Graduations sur les paliers RÉELS : en log, matplotlib affiche sinon 4×10¹, ce qui
# oblige le lecteur à traduire mentalement l'axe le plus important du graphique.
all_tiers = sorted(SUMMARY["utilisateurs"].unique())
ax.set_xticks(all_tiers)
ax.set_xticklabels([f"{int(t):,}".replace(",", " ") for t in all_tiers])
ax.minorticks_off()
ax.set_xlabel("utilisateurs d'entraînement (échelle log)")
ax.set_ylabel("top-1 répertoire (test)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Courbe de scaling — même test, même baseline, mêmes hyperparamètres")
ax.legend(frameon=False, fontsize=9)
_clean(ax)
plt.tight_layout(); _save(fig, "01_courbe_scaling"); plt.show()

# ------------------------------------------------- 2. gain marginal + IC
# La question « où ça sature » se lit ici, pas sur la courbe : une barre dont
# l'intervalle traverse 0 est un palier qui n'a rien apporté.
for proto, df in STEPS_TABLE.items():
    if df.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 3.9))
    xs = np.arange(len(df))
    colors = [BLUE if r["significatif"] else INK_MUTED for _, r in df.iterrows()]
    ax.bar(xs, df["delta"], width=0.6, color=colors, zorder=2)
    ax.errorbar(xs, df["delta"],
                yerr=np.vstack([df["delta"] - df["lo"], df["hi"] - df["delta"]]),
                fmt="none", ecolor=INK_SOFT, capsize=4, linewidth=1.2, zorder=3)
    ax.axhline(0, color=INK_SOFT, linewidth=1)
    ax.set_xticks(xs, [f"{int(a)}→{int(b)}" for a, b in zip(df["de"], df["à"])])
    ax.set_ylabel("gain top-1 (apparié)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:+.1%}")
    ax.set_title(f"[{proto}] Gain marginal d'un palier au suivant, IC 95% apparié\n"
                 "(gris = intervalle traversant 0 : gain non démontré)", fontsize=11)
    _clean(ax)
    plt.tight_layout(); _save(fig, f"02_gain_marginal_{proto}"); plt.show()

# ------------------------------------------------- 3. démarrage à froid
# Une identité par palier ; l'axe x est la longueur d'historique disponible.
fig, ax = plt.subplots(figsize=(9, 4.8))
shown = [n for n in ORDER if RESULTS[n]["protocol"] == PROTOCOLS[0]]
fr = sorted(float(k) for k in RESULTS[shown[0]]["test"]["by_fraction"])
for i, name in enumerate(shown):
    ys = [RESULTS[name]["test"]["by_fraction"][f"{f:.2f}"]["metrics"]["top1_seen"] for f in fr]
    ax.plot(fr, ys, color=SERIES[i % len(SERIES)], linewidth=2, marker="o", markersize=5,
            label=f"{RESULTS[name]['n_users']} users")
base_ys = [RESULTS[shown[0]]["test"]["by_fraction"][f"{f:.2f}"]["baseline"] for f in fr]
ax.plot(fr, base_ys, color=INK_SOFT, linewidth=1.6, linestyle="--", label="baseline")
ax.set_xlabel("fraction de l'historique donnée en contexte")
ax.set_ylabel("top-1 répertoire (test)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Démarrage à froid : l'échelle paie-t-elle quand l'historique est court ?")
ax.legend(frameon=False, fontsize=9, ncol=2)
_clean(ax)
plt.tight_layout(); _save(fig, "03_demarrage_a_froid"); plt.show()

# ------------------------------------------------- 4. par groupe
if not GROUPS_DF.empty:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    cols = [c for c in GROUPS_DF.columns if c.startswith("G") and not c.endswith(" n")]
    sub = GROUPS_DF[GROUPS_DF.index.isin(shown)]
    for i, c in enumerate(sorted(cols)):
        ax.plot(sub["utilisateurs"], sub[c], color=SERIES[i % len(SERIES)], linewidth=2,
                marker="o", markersize=5, label=c)
    ax.set_xscale("log")
    ax.set_xticks(sorted(sub["utilisateurs"].unique()))
    ax.set_xticklabels([f"{int(t):,}".replace(",", " ")
                        for t in sorted(sub["utilisateurs"].unique())])
    ax.minorticks_off()
    ax.set_xlabel("utilisateurs d'entraînement (échelle log)")
    ax.set_ylabel("top-1 répertoire (test)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("Qui profite de l'échelle ? Accuracy par groupe de comportement")
    ax.legend(frameon=False, fontsize=9, ncol=4)
    _clean(ax)
    plt.tight_layout(); _save(fig, "04_par_groupe"); plt.show()

# ------------------------------------------------- 5. convergence
fig, ax = plt.subplots(figsize=(9, 4.8))
for i, name in enumerate(shown):
    h = RESULTS[name]["training"]["history"]
    if not h:
        continue
    ax.plot([e["step"] for e in h], [e["top1_seen"] for e in h],
            color=SERIES[i % len(SERIES)], linewidth=2, marker="o", markersize=3.5,
            label=f"{RESULTS[name]['n_users']} users")
    b = RESULTS[name]["training"]["best_step"]
    if b is not None:
        ax.plot([b], [max(e["top1_seen"] for e in h)], marker="o", markersize=8,
                markerfacecolor="none", markeredgecolor=SERIES[i % len(SERIES)],
                markeredgewidth=1.8)
ax.set_xlabel("pas d'optimisation")
ax.set_ylabel("top-1 répertoire (validation)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Convergence à budget de pas identique (cercle = pic retenu)")
ax.legend(frameon=False, fontsize=9, ncol=2)
_clean(ax)
plt.tight_layout(); _save(fig, "05_convergence"); plt.show()

# ------------------------------------------------- 6. coût
# L'axe qu'on oublie toujours dans une étude de scalabilité : ce que le gain coûte.
fig, ax = plt.subplots(figsize=(9, 4.2))
for proto in PROTOCOLS:
    sub = SUMMARY[SUMMARY["protocole"] == proto].sort_values("utilisateurs")
    ax.plot(sub["min_GPU"], sub["écart"], color=COLOR_OF_PROTO[proto], linewidth=2,
            marker="o", markersize=6, label=proto)
    for _, r in sub.iterrows():
        ax.annotate(f"{int(r['utilisateurs'])}", xy=(r["min_GPU"], r["écart"]),
                    xytext=(0, 8), textcoords="offset points", ha="center",
                    fontsize=8.5, color=INK_SOFT)
ax.axhline(0, color=INK_SOFT, linewidth=1)
ax.set_xlabel("minutes GPU du palier (entraînement + évaluations)")
ax.set_ylabel("gain sur la baseline")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:+.0%}")
ax.set_title("Ce que le gain coûte — points annotés par nombre d'utilisateurs")
ax.legend(frameon=False, fontsize=9)
_clean(ax)
plt.tight_layout(); _save(fig, "06_cout"); plt.show()

print(f"Figures PNG écrites dans {FIG_DIR}")


## Notes

### Ce que ce notebook garantit

| Risque | Traitement |
|---|---|
| Paliers non emboîtés (« plus de données » confondu avec « d'autres données ») | Chaque palier est une **tranche initiale** de `ALL_TRAIN`, lue une seule fois. Le fichier `merged/train.jsonl` est déjà mélangé et les 11 jours y sont uniformément répartis, y compris dans ses 1 000 premières lignes. |
| Softmax de taille variable | `VOCAB` = union train ∪ test, construit **avant** le premier palier. Un `cell_id` qu'un petit palier ne voit jamais reste à son initialisation — et c'est la ligne « cellules rares » qui le mesure, pas la taille du softmax. |
| Adaptateurs LoRA empilés d'un palier sur l'autre | `new_model()` recharge la base et l'enveloppe à neuf ; `get_peft_model` modifie le modèle **en place**, donc le rappeler sur un modèle déjà enveloppé ferait démarrer le palier N sur les poids du palier N−1. |
| Règle d'arrêt différente selon le palier | En `iso_steps`, la validation est cadencée **en pas**, pas en epochs : une epoch vaut 67 pas au palier 500 et 2 815 au palier 21 000, donc « 4 epochs sans progrès » serait 42 fois plus permissif en haut qu'en bas. |
| Bruit de validation différent selon le palier | `val_subsample` prend les **mêmes utilisateurs** partout (tranche initiale du fichier, incluse dans tous les paliers). |
| Bruit d'initialisation pris pour un effet d'échelle | `set_seed(CONFIG["seed"])` au début de `new_model()`. |
| VRAM saturée au 4ᵉ palier | `del` + `gc.collect()` + `empty_cache()` en fin de `run_tier`. |
| Interpréter un écart qui n'existe pas | Bootstrap **apparié** sur les utilisateurs de test, communs à tous les paliers. Un gain dont l'intervalle contient 0 est affiché en gris sur le graphique 2. |

### Les deux protocoles, et lequel citer

`iso_steps` est **la** courbe de scaling : tous les paliers reçoivent le même
budget de pas, le même warmup et le même horizon cosine, donc la seule variable est
le nombre d'utilisateurs distincts. C'est celle à présenter.

`iso_epochs` répond à une autre question — « et si on laisse chaque palier
converger ? » — et reste comparable aux runs des présentations précédentes. Son
coût croît **linéairement** avec le palier : à 21 000 utilisateurs, 5 epochs valent
déjà ~14 000 pas, soit dix fois le budget `iso_steps`. C'est pourquoi il n'est
appliqué par défaut qu'à un seul palier. Pour l'étendre, ajoutez les paliers voulus
dans `SCALE_CONFIG["runs"]` **et** relancez la calibration : la projection vous
dira avant de partir si ça tient dans la session.

### Ce qu'il ne faut PAS régler pour aller plus vite

| Fausse bonne idée | Ce qui se passe vraiment |
|---|---|
| Monter le batch d'entraînement | À learning rate constant, un batch 4× plus grand donne 4× moins de pas par epoch. Le modèle apprend autre chose, pas la même chose plus vite — et les paliers déjà calculés deviennent incomparables. |
| Baisser `max_steps` en cours de route | `max_steps` est aussi l'**horizon du scheduler cosine**. Un palier à 1 500 pas et un autre à 800 ne sont pas sur la même courbe : il faut refaire **tous** les paliers. |
| Baisser `val_subsample` pour un seul palier | La règle d'arrêt cesse d'être la même partout. Si vous y touchez, touchez-y pour tous les paliers, et refaites-les tous. |
| Mettre `diagnostics_tiers=[]` | C'est en revanche un réglage **sain** : il ne change ni l'entraînement ni le score de référence, seulement les ablations, qui n'ont besoin d'être comparées qu'entre les extrêmes de l'échelle. |

### Reprise après une déconnexion Colab

Relancer la cellule BOUCLE, c'est tout. Les jobs dont le JSON porte
`"status": "complete"` sont relus et sautés ; un job interrompu pendant son
entraînement (`"status": "trained"`, ou pas de JSON) est refait entièrement. Les
jobs s'exécutent du plus petit palier au plus grand : ce qui manque après une
coupure est toujours le **haut** de la courbe, jamais le bas, donc ce qui est déjà
calculé reste interprétable tel quel.

Les cellules SYNTHÈSE, DÉTAILS et GRAPHIQUES ne lisent que les JSON : elles
tournent sur une session neuve, sans GPU.

### Les limites à annoncer avec les résultats

1. **Le monde est petit.** 369 `cell_id` sur 113 sites physiques. La couverture du
   vocabulaire de test est déjà de 98,9 % au palier 500 et de 100 % à partir de
   5 000 : au-delà, l'échelle ne peut plus rien apporter par « découverte de
   nouvelles cellules », seulement par meilleure estimation des transitions. Une
   saturation de la courbe est donc le résultat attendu, pas un échec — et le
   tableau « cellules rares » est ce qui permet de le dire proprement.
2. **Un utilisateur = une seule journée**, et les utilisateurs sont disjoints d'un
   jour à l'autre. Le modèle ne peut apprendre que des régularités de population,
   jamais la personnalisation d'un client suivi dans le temps. Ce que mesure cette
   courbe, c'est la valeur d'une **population** plus grande, pas d'un **historique**
   plus long.
3. **Un seul tirage par palier.** Chaque point est un entraînement, avec une seule
   graine. L'IC apparié couvre l'échantillonnage du test, pas la variance
   d'entraînement. Pour la mesurer, refaites un palier intermédiaire avec
   `CONFIG["seed"]` différent et un `RESULTS_DIR` distinct : l'écart entre les deux
   graines est le plancher de bruit sous lequel aucun écart entre paliers n'est
   interprétable.
4. **Les groupes sont réajustés à chaque palier**, comme il se doit (un groupe doit
   être dérivable des seules données du palier). Les groupes étant numérotés par
   mobilité croissante, `G0` reste « le plus sédentaire », mais ne comparez pas les
   effectifs de groupe au point de pourcentage près entre deux paliers.
